# VERSÃO LIMPA - PIPELINE OFICIAL

In [1]:
!pip install requests beautifulsoup4 pandas lxml -q

In [2]:
import requests
import json
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime
import pandas as pd

## Uso da API Senado Federal

Nesta etapa será definida a URN da norma jurídica e o endpoint principal da API de Dados Abertos do Senado.

A URN será usada como identificador único da norma durante todo o fluxo.  
A partir dela será possível consultar os metadados oficiais e depois relacionar esses dados com o texto integral extraído de outras fontes oficiais.


In [3]:
URN_ALVO = "urn:lex:br:federal:lei:1990-12-11;8112"
VERSAO_API = 3

URL_API_SENADO_URN = "https://legis.senado.leg.br/dadosabertos/legislacao/urn.json"

## Configuração da sessão e padronização das requisições

Nesta etapa será criada uma sessão HTTP reutilizável para evitar repetição de cabeçalhos e melhorar a estabilidade das requisições.

O objetivo é padronizar o acesso aos sites oficiais e preparar o ambiente para consultar a API do Senado, o LexML e as páginas HTML com o inteiro teor.

In [4]:
from urllib.parse import urljoin
from datetime import datetime

session = requests.Session() #requests.get apenas nao funcionou
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/html, application/xhtml+xml"
})

## Funções utilitárias para coleta e tratamento dos dados

Nesta etapa serão definidas funções auxiliares para:

- obter respostas JSON;
- obter páginas HTML;
- limpar texto bruto;
- localizar campos dentro de estruturas JSON aninhadas.

Essas funções serão reutilizadas nas próximas etapas para deixar o notebook mais limpo, organizado e fácil de manter.

In [5]:
def get_json(url: str, params: dict | None = None, timeout: int = 30):
    resp = session.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

#Usado para obter resposta com texto integral
def get_html(url: str, params: dict | None = None, timeout: int = 30) -> str:
    resp = session.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding or "utf-8"
    return resp.text


def limpar_texto(texto: str) -> str:
    if not texto:
        return ""
    texto = texto.replace("\ufeff", " ")
    texto = re.sub(r"\r", "\n", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"[ \t]{2,}", " ", texto)
    return texto.strip()


def buscar_todos_campos(obj, chave_alvo):
    resultados = []

    if isinstance(obj, dict):
        for chave, valor in obj.items():
            if chave == chave_alvo:
                resultados.append(valor)
            resultados.extend(buscar_todos_campos(valor, chave_alvo))
    elif isinstance(obj, list):
        for item in obj:
            resultados.extend(buscar_todos_campos(item, chave_alvo))

    return resultados


def buscar_primeiro_campo(obj, chave_alvo, default=None):
    encontrados = buscar_todos_campos(obj, chave_alvo)
    return encontrados[0] if encontrados else default

## Coleta de metadados oficiais por meio da API do Senado

A API do Senado permite obter metadados da norma, como ementa, número, ano, data de assinatura e informações de publicação.

Histórico com alterações também podem ser obtidas. Interessante em uso futuro para construção de grafos.

Nesta etapa será feita a consulta oficial usando a URN como chave principal, e também será criado um tratamento inicial para organizar os campos mais relevantes da resposta.

In [6]:
import time
import requests

def baixar_metadados_api_senado_por_urn(urn, versao_api, max_tentativas=5, timeout=30):
    """
    Consulta a API oficial do Senado com tentativas automáticas.
    Trata falhas temporárias de rede, DNS e erros 5xx do servidor.
    """
    url = f"https://legis.senado.leg.br/dadosabertos/legislacao/urn.json?urn={urn}&v={versao_api}"
    ultimo_erro = None

    for tentativa in range(1, max_tentativas + 1):
        try:
            resposta = requests.get(url, timeout=timeout)
            resposta.raise_for_status()
            return resposta.json()

        except requests.exceptions.RequestException as e:
            ultimo_erro = e
            print(f"Tentativa {tentativa}/{max_tentativas} falhou: {e}")

            if tentativa < max_tentativas:
                time.sleep(2 * tentativa)

    raise RuntimeError(
        f"Falha ao consultar a API oficial do Senado após {max_tentativas} tentativas. "
        f"Último erro: {ultimo_erro}"
    )


def extrair_metadados_api(raw_api: dict) -> dict:
    """
    Extrai os metadados mínimos e estáveis da resposta da API do Senado.
    Mantém compatibilidade com as fases posteriores do notebook.
    """
    if not isinstance(raw_api, dict):
        raise TypeError("raw_api precisa ser um dict retornado pela API do Senado.")

    def primeiro_preenchido_lista(*listas):
        for lista in listas:
            if isinstance(lista, list):
                for item in lista:
                    if item not in (None, "", [], {}):
                        return item
        return None

#Melhorar case sensitive com regex futuramente
    urn = primeiro_preenchido_lista(
        buscar_todos_campos(raw_api, "urn"),
        buscar_todos_campos(raw_api, "URN")
    )

    ementa = primeiro_preenchido_lista(
        buscar_todos_campos(raw_api, "ementa"),
        buscar_todos_campos(raw_api, "Ementa"),
        buscar_todos_campos(raw_api, "descricaoIdentificacao"),
        buscar_todos_campos(raw_api, "DescricaoIdentificacao"),
        buscar_todos_campos(raw_api, "textoEmenta"),
        buscar_todos_campos(raw_api, "TextoEmenta")
    )

    numero = primeiro_preenchido_lista(
        buscar_todos_campos(raw_api, "numero"),
        buscar_todos_campos(raw_api, "Numero")
    )

    ano = primeiro_preenchido_lista(
        buscar_todos_campos(raw_api, "ano"),
        buscar_todos_campos(raw_api, "Ano")
    )

    dataassinatura = primeiro_preenchido_lista(
        buscar_todos_campos(raw_api, "dataAssinatura"),
        buscar_todos_campos(raw_api, "dataassinatura"),
        buscar_todos_campos(raw_api, "DataAssinatura"),
        buscar_todos_campos(raw_api, "data"),
        buscar_todos_campos(raw_api, "Data")
    )

    titulo_norma = primeiro_preenchido_lista(
        buscar_todos_campos(raw_api, "titulo"),
        buscar_todos_campos(raw_api, "Titulo"),
        buscar_todos_campos(raw_api, "tituloNorma"),
        buscar_todos_campos(raw_api, "TituloNorma")
    )

    url_documento = primeiro_preenchido_lista(
        buscar_todos_campos(raw_api, "urlDocumento"),
        buscar_todos_campos(raw_api, "UrlDocumento"),
        buscar_todos_campos(raw_api, "linkInteiroTeor"),
        buscar_todos_campos(raw_api, "LinkInteiroTeor"),
        buscar_todos_campos(raw_api, "href")
    )

    publicacoes = []
    for chave in ["publicacao", "publicacoes", "Publicacao", "Publicacoes"]:
        encontrados = buscar_todos_campos(raw_api, chave)
        for item in encontrados:
            if isinstance(item, list):
                publicacoes.extend(item)
            elif isinstance(item, dict):
                publicacoes.append(item)

    # Remove duplicações simples preservando ordem
    publicacoes_unicas = []
    vistos = set()
    for pub in publicacoes:
        marcador = json.dumps(pub, ensure_ascii=False, sort_keys=True) if isinstance(pub, dict) else str(pub)
        if marcador not in vistos:
            vistos.add(marcador)
            publicacoes_unicas.append(pub)

    return {
        "urn": str(urn) if urn is not None else (URN_ALVO if "URN_ALVO" in globals() else None),
        "ementa": str(ementa) if ementa is not None else None,
        "numero": str(numero) if numero is not None else None,
        "ano": str(ano) if ano is not None else None,
        "dataassinatura": str(dataassinatura) if dataassinatura is not None else None,
        "titulo_norma": str(titulo_norma) if titulo_norma is not None else None,
        "publicacoes": publicacoes_unicas,
        "urlDocumento": str(url_documento) if url_documento is not None else None
    }

## Execução da consulta oficial da API

Nesta etapa será executada a requisição à API do Senado e os metadados serão armazenados em variáveis prontas para uso nas próximas fases.

A intenção aqui é validar se a URN foi reconhecida corretamente e se a API devolveu os dados esperados da norma.

In [7]:
raw_api_senado = baixar_metadados_api_senado_por_urn(URN_ALVO, VERSAO_API)
metadados_api = extrair_metadados_api(raw_api_senado)

print("URN:", metadados_api["urn"])
print("Ementa:", metadados_api["ementa"])
print("Qtd. publicações:", len(metadados_api["publicacoes"]))
print("urlDocumento:", metadados_api["urlDocumento"])

URN: urn:lex:br:federal:lei:1990-12-11;8112
Ementa: Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais
Qtd. publicações: 4
urlDocumento: https://normas.leg.br/?urn=urn:lex:br:federal:lei:1990-12-11;8112


In [8]:
url = "https://legis.senado.leg.br/dadosabertos/legislacao/urn.json?urn=urn:lex:br:federal:lei:1990-12-11;8112&v=3"
r = requests.get(url, timeout=30) #utilizado o sessions.get em vez de apenas requests.get
print(r.status_code)
print(r.text[:500])

200
{"DetalheDocumento":{"noNamespaceSchemaLocation":"https://legis.senado.leg.br/dadosabertos/dados/DetalheDocumentov3.xsd","Metadados":{"Versao":"19/04/2026 01:07:58","VersaoServico":"3","DataVersaoServico":"2017-02-01","DescricaoDataSet":"Obtém detalhes de uma Norma Jurídica por meio da sua URN."},"documentos":{"documento":[{"id":"549988","identificacao":{"tipo":"LEI-n","descricao":"Lei Numerada","numero":"8112","norma":"LEI-8112-1990-12-11","normaNome":"Lei nº 8.112 de 11/12/1990","dataassinatur


## Resolução da URN e descoberta das fontes oficiais pelo LexML

A URN também será usada para consultar o LexML, que funciona como um resolvedor de fontes ligadas à mesma norma.

O JSON retornado na API Senado traz apenas metadados, não tem texto integral da norma, necessário ao repasse de contexto ao LLM da aplicação RAG.

Nesta etapa serão identificados os links oficiais mais úteis para obtenção do inteiro teor, priorizando:

- Câmara dos Deputados;
- Senado Federal;
- Planalto, quando disponível.

O objetivo é escolher links diretos e evitar links intermediários que prejudiquem a extração.

In [9]:

import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def criar_estrutura_links_oficiais(urn: str, erro_lexml: str | None = None) -> dict:
    """
    Estrutura padrão dos links oficiais da norma.
    """
    return {
        "lexml": f"https://www.lexml.gov.br/urn/{urn}",
        "camara": None,
        "senado": None,
        "planalto": None,
        "candidatos": {
            "camara_publicacao": None,
            "camara_metadado": None,
            "senado_publicacao": None,
            "senado_metadado": None,
            "planalto": None,
            "senado_api": None
        },
        "erro_lexml": erro_lexml
    }


def extrair_links_lexml(urn, max_tentativas=3, timeout=20, verbose=False):
    """
    Consulta o LexML para descobrir links oficiais úteis da norma.

    Comportamento desta versão:
    - tenta recuperar os links sem poluir a saída do notebook;
    - em caso de falha do LexML, retorna estrutura vazia com erro registrado;
    - não interrompe o pipeline principal só por instabilidade temporária do LexML.
    """
    url_lexml = f"https://www.lexml.gov.br/urn/{urn}"
    ultimo_erro = None

    for tentativa in range(1, max_tentativas + 1):
        try:
            resposta = session.get(url_lexml, timeout=timeout) if "session" in globals() else requests.get(url_lexml, timeout=timeout)
            resposta.raise_for_status()

            soup = BeautifulSoup(resposta.text, "html.parser")
            links_norma = criar_estrutura_links_oficiais(urn)

            itens = soup.find_all(class_="list-group-item")

            for item in itens:
                texto = item.get_text(" ", strip=True).lower()
                link_tag = item.find("a", href=True)

                if not link_tag:
                    continue

                url = urljoin(url_lexml, link_tag["href"])

                # Planalto
                if "presidência da república" in texto or "presidencia da republica" in texto or "planalto" in texto or "planalto" in url:
                    links_norma["candidatos"]["planalto"] = url
                    if not links_norma["planalto"]:
                        links_norma["planalto"] = url

                # Câmara
                elif "câmara dos deputados" in texto or "camara dos deputados" in texto:
                    if "publicacaooriginal" in url:
                        links_norma["candidatos"]["camara_publicacao"] = url
                        if not links_norma["camara"]:
                            links_norma["camara"] = url
                    else:
                        links_norma["candidatos"]["camara_metadado"] = url
                        if not links_norma["camara"]:
                            links_norma["camara"] = url

                # Senado
                elif "senado federal" in texto:
                    if "/publicacao/" in url:
                        links_norma["candidatos"]["senado_publicacao"] = url
                        if not links_norma["senado"]:
                            links_norma["senado"] = url
                    else:
                        links_norma["candidatos"]["senado_metadado"] = url
                        if not links_norma["senado"]:
                            links_norma["senado"] = url

            # Regras finais de preferência
            if links_norma["candidatos"]["camara_publicacao"]:
                links_norma["camara"] = links_norma["candidatos"]["camara_publicacao"]

            if links_norma["candidatos"]["senado_publicacao"]:
                links_norma["senado"] = links_norma["candidatos"]["senado_publicacao"]

            return links_norma

        except requests.exceptions.RequestException as e:
            ultimo_erro = e
            if verbose:
                print(f"Tentativa {tentativa}/{max_tentativas} no LexML falhou: {e}")

            if tentativa < max_tentativas:
                time.sleep(min(2 * tentativa, 5))

    # Não explode o pipeline: devolve estrutura consistente com o erro registrado
    return criar_estrutura_links_oficiais(urn, erro_lexml=str(ultimo_erro))

## Execução da coleta dos links oficiais

Nesta etapa será executada a leitura da página do LexML para verificar quais links oficiais foram encontrados para a norma.

Esse passo é importante porque define de onde será obtido o inteiro teor que será unido aos metadados já coletados pela API do Senado.

In [10]:
links_oficiais = extrair_links_lexml(URN_ALVO)

print(json.dumps(links_oficiais, indent=2, ensure_ascii=False))

{
  "lexml": "https://www.lexml.gov.br/urn/urn:lex:br:federal:lei:1990-12-11;8112",
  "camara": "http://www2.camara.gov.br/legin/fed/lei/1990/lei-8112-11-dezembro-1990-322161-publicacaooriginal-1-pl.html",
  "senado": "http://legis.senado.leg.br/norma/549988/publicacao/15722628",
  "planalto": "http://legislacao.planalto.gov.br/legisla/legislacao.nsf/websearch?openagent&tipo=LEI&codigo=8.112&ementa=2&data=19901211",
  "candidatos": {
    "camara_publicacao": "http://www2.camara.gov.br/legin/fed/lei/1990/lei-8112-11-dezembro-1990-322161-publicacaooriginal-1-pl.html",
    "camara_metadado": "http://www2.camara.gov.br/legin/fed/lei/1990/lei-8112-11-dezembro-1990-322161-norma-pl.html",
    "senado_publicacao": "http://legis.senado.leg.br/norma/549988/publicacao/15722628",
    "senado_metadado": "http://legis.senado.leg.br/norma/549988",
    "planalto": "http://legislacao.planalto.gov.br/legisla/legislacao.nsf/websearch?openagent&tipo=LEI&codigo=8.112&ementa=2&data=19901211",
    "senado_ap

## - Acabamento e limpeza do texto integral

Na fase anterior foi possível coletar o inteiro teor da norma e unificar o resultado com os metadados oficiais.

Nesta fase será feito o acabamento do texto, removendo cabeçalhos de portal, trechos de navegação e avisos que não pertencem ao corpo principal da norma.

O objetivo é deixar o texto mais limpo para as próximas etapas do projeto, especialmente a segmentação em artigos e a futura indexação em banco vetorial.

In [11]:
def aparar_inicio_texto_norma(texto: str) -> str:
    """
    Remove cabeçalhos de portal e tenta iniciar no ponto correto da norma.

    Regra principal:
    - se encontrar o título da norma principal, mantém o título;
    - se não encontrar, tenta começar em 'O PRESIDENTE DA REPÚBLICA';
    - se ainda não encontrar, tenta começar no Art. 1º.
    """
    texto = limpar_texto(texto)

    numero_norma = None

    if "metadados_api" in globals() and isinstance(metadados_api, dict):
        numero_norma = metadados_api.get("numero")

    if numero_norma:
        numero_esc = re.escape(str(numero_norma)).replace("\\.", r"\.")
        padrao_titulo = rf'LEI\s*N[º°o]\s*{numero_esc}\b'
        match_titulo = re.search(padrao_titulo, texto)
        if match_titulo:
            texto = texto[match_titulo.start():]
            return limpar_texto(texto)

    match_presidente = re.search(r'O\s+PRESIDENTE\s+DA\s+REP[ÚU]BLICA', texto)
    if match_presidente:
        texto = texto[match_presidente.start():]
        return limpar_texto(texto)

    match_art1 = re.search(r'Art\.\s*1[º°]?', texto)
    if match_art1:
        texto = texto[match_art1.start():]
        return limpar_texto(texto)

    return limpar_texto(texto)


def aparar_fim_texto_norma(texto: str) -> str:
    """
    Remove possíveis trechos finais de navegação do portal.
    """
    texto = limpar_texto(texto)

    padroes_fim = [
        r'\nVoltar ao topo.*$',
        r'\nPágina atualizada em.*$',
        r'\nExibir outro ato.*$',
        r'\nImprimir.*$'
    ]

    for padrao in padroes_fim:
        texto = re.sub(padrao, '', texto, flags=re.DOTALL)

    return limpar_texto(texto)


def reformatar_texto_legislativo(texto: str) -> str:
    """
    Reorganiza o texto para ficar legível e evitar que referências internas
    em minúsculo ('art. 41', 'art. 93') virem novos artigos.
    """
    texto = limpar_texto(texto)

    numero_norma = None
    ementa_norma = None

    if "metadados_api" in globals() and isinstance(metadados_api, dict):
        numero_norma = metadados_api.get("numero")
        ementa_norma = metadados_api.get("ementa")

    # Normaliza espaços gerais
    texto = re.sub(r'\s+', ' ', texto).strip()

    # Se houver número da norma, tenta separar corretamente o título
    if numero_norma:
        numero_esc = re.escape(str(numero_norma)).replace("\\.", r"\.")
        texto = re.sub(
            rf'(LEI\s*N[º°o]\s*{numero_esc}\b[^A-Z]{{0,120}}?\d{{4}})\s+',
            r'\1\n',
            texto
        )

    # Se houver ementa conhecida, força a separação correta
    if ementa_norma:
        ementa_esc = re.escape(ementa_norma.strip())

        texto = re.sub(
            rf'(\b\d{{4}})\s+({ementa_esc})',
            r'\1\n\2',
            texto
        )

        texto = re.sub(
            rf'({ementa_esc})\s+(O\s+PRESIDENTE\s+DA\s+REP[ÚU]BLICA)',
            r'\1\n\n\2',
            texto
        )

    # Mantém quebra antes do preâmbulo
    texto = re.sub(
        r'\s*(O\s+PRESIDENTE\s+DA\s+REP[ÚU]BLICA)',
        r'\n\n\1',
        texto
    )

    # Quebras estruturais grandes com divisões em numeração algarismos romanos
    grandes_blocos = [
        r'(TÍTULO\s+[IVXLCDM]+)',
        r'(CAPÍTULO\s+[IVXLCDM]+)',
        r'(Seção\s+[IVXLCDM]+)',
        r'(Subseção\s+[IVXLCDM]+)'
    ]
    for padrao in grandes_blocos:
        texto = re.sub(rf'\s*{padrao}', r'\n\n\1', texto)

    # IMPORTANTE:
    # quebra antes de artigos somente quando vier com "Art." em maiúsculo
    # para não transformar "art. 41" interno em novo artigo
    texto = re.sub(r'\s*(Art\.\s*\d+[A-Z\-]*[º°]?)', r'\n\1', texto)
#De forma semelhante para paragrafos e incisos
    texto = re.sub(r'\s*(Parágrafo único\.)', r'\n\1', texto)
    texto = re.sub(r'\s*(§\s*\d+[º°]?)', r'\n\1', texto)
    texto = re.sub(r'\s*([IVXLCDM]+\s*-\s)', r'\n\1', texto)

    # Ajustes para títulos colados
    texto = re.sub(
        r'(TÍTULO\s+[IVXLCDM]+)\s+(CAPÍTULO\s+[IVXLCDM]+)',
        r'\1\n\2',
        texto
    )
    texto = re.sub(
        r'(CAPÍTULO\s+[IVXLCDM]+)\s+(Seção\s+[IVXLCDM]+)',
        r'\1\n\2',
        texto
    )

    texto = re.sub(r'\n{3,}', '\n\n', texto)

    return texto.strip()


def limpar_texto_norma_final(texto: str) -> str:
    """
    Pipeline final de limpeza do texto legislativo.
    """
    texto = limpar_texto(texto)
    texto = aparar_inicio_texto_norma(texto)
    texto = aparar_fim_texto_norma(texto)
    texto = reformatar_texto_legislativo(texto)
    return texto.strip()

## Recomposição do cabeçalho canônico da norma

Na etapa anterior foi observado que o bloco principal extraído da Câmara dos Deputados continha o corpo da norma, mas nem sempre preservava no mesmo bloco o título formal completo da lei.

Por esse motivo, nesta etapa foi implementada uma recomposição do cabeçalho canônico da norma com base nos metadados oficiais já coletados pela API do Senado.

A lógica aplicada passa a acrescentar, no início do texto consolidado:

- o título formal da norma;
- a ementa;
- o preâmbulo e o corpo normativo já extraídos da fonte principal.

Com isso, o texto final preserva melhor a identificação jurídica da norma e fica mais adequado para as próximas fases do projeto, especialmente a transformação em JSON, a segmentação em partes menores e a futura indexação em banco vetorial.

In [12]:
MESES_PTBR = {
    1: "JANEIRO",
    2: "FEVEREIRO",
    3: "MARÇO",
    4: "ABRIL",
    5: "MAIO",
    6: "JUNHO",
    7: "JULHO",
    8: "AGOSTO",
    9: "SETEMBRO",
    10: "OUTUBRO",
    11: "NOVEMBRO",
    12: "DEZEMBRO"
}


def montar_titulo_canonico_norma() -> str | None:
    """
    Monta um título canônico da norma a partir dos metadados da API.
    Ex.: LEI Nº 8.112, DE 11 DE DEZEMBRO DE 1990
    """
    if "metadados_api" not in globals() or not isinstance(metadados_api, dict):
        return None

    numero = metadados_api.get("numero")
    data = metadados_api.get("dataassinatura")

    if not numero:
        return None

    # tenta usar a data para montar o título completo
    if data:
        match = re.match(r"(\d{2})/(\d{2})/(\d{4})", str(data))
        if match:
            dia = int(match.group(1))
            mes = int(match.group(2))
            ano = match.group(3)
            nome_mes = MESES_PTBR.get(mes)

            if nome_mes:
                return f"LEI Nº {numero}, DE {dia} DE {nome_mes} DE {ano}"

    # fallback
    return f"LEI Nº {numero}"


def enriquecer_texto_com_cabecalho_canonico(texto: str) -> str:
    """
    Acrescenta título e ementa no início do texto, se eles ainda não estiverem presentes.
    """
    texto = limpar_texto(texto)

    titulo = montar_titulo_canonico_norma()
    ementa = None

    if "metadados_api" in globals() and isinstance(metadados_api, dict):
        ementa = metadados_api.get("ementa")

    inicio_busca = texto[:1200].lower()
    partes = []

    if titulo and titulo.lower() not in inicio_busca:
        partes.append(titulo)

    if ementa and ementa.strip() and ementa.lower() not in inicio_busca:
        partes.append(ementa.strip())

    partes.append(texto)

    texto_final = "\n".join(partes)
    texto_final = re.sub(r"\n{3,}", "\n\n", texto_final).strip()

    return texto_final

## Conteudo textual e html do inteiro teor da norma - Origem Camara

A Câmara dos Deputados disponibiliza páginas HTML com o corpo textual da norma.

Nesta etapa será feita a tentativa de localizar o bloco principal do inteiro teor dentro da página da Câmara, preservando:

1. o texto limpo;
2. o HTML bruto do bloco encontrado;
3. o tamanho final do conteúdo extraído.

Essa origem será priorizada quando o link direto de publicação original estiver disponível.

In [13]:
def extrair_texto_camara(url: str) -> dict:
    """
    Extrai o inteiro teor da Câmara dos Deputados, limpa o texto
    e acrescenta cabeçalho canônico da norma.
    """
    html = get_html(url)
    soup = BeautifulSoup(html, "html.parser")

    # Prioridade máxima para o seletor que já funcionou melhor
    bloco = soup.find("div", class_="texto")

    # Fallbacks
    if bloco is None:
        fallbacks = [
            ("div", {"class": "textoNorma"}),
            ("div", {"class": "documentContent"}),
            ("div", {"id": "content"})
        ]

        melhor_bloco = None
        melhor_texto = ""

        for tag, attrs in fallbacks:
            candidato = soup.find(tag, attrs=attrs)
            if candidato:
                texto_candidato = limpar_texto(candidato.get_text(" ", strip=True))
                if len(texto_candidato) > len(melhor_texto):
                    melhor_bloco = candidato
                    melhor_texto = texto_candidato

        bloco = melhor_bloco

    if bloco is None:
        raise ValueError("Não foi possível localizar o corpo do texto na Câmara.")

    html_bloco = str(bloco)
    texto_bruto = limpar_texto(bloco.get_text(" ", strip=True))
    texto_corpo_limpo = limpar_texto_norma_final(texto_bruto)
    texto_com_cabecalho = enriquecer_texto_com_cabecalho_canonico(texto_corpo_limpo)

    return {
        "fonte": "camara",
        "url": url,
        "html": html_bloco,
        "texto_bruto": texto_bruto,
        "texto_corpo_limpo": texto_corpo_limpo,
        "texto": texto_com_cabecalho,
        "tamanho_texto_bruto": len(texto_bruto),
        "tamanho_texto_corpo_limpo": len(texto_corpo_limpo),
        "tamanho_texto": len(texto_com_cabecalho)
    }

## Conteudo textual e html do inteiro teor da norma - Origem Senado

O Senado também disponibiliza o inteiro teor da norma em página HTML.

Nesta etapa será feita a extração do conteúdo principal do texto legislativo a partir da estrutura da página do Senado, permitindo comparar a origem obtida pela Câmara com a origem obtida pelo Senado.

In [14]:
def extrair_texto_senado(url: str) -> dict:
    """
    Extrai o inteiro teor do Senado Federal, limpa o texto
    e acrescenta cabeçalho canônico da norma quando necessário.
    """
    html = get_html(url)
    soup = BeautifulSoup(html, "html.parser")

    seletores = [
        ("div", {"id": "conteudoPrincipal"}),
        ("main", {}),
        ("article", {})
    ]

    melhor_bloco = None
    melhor_texto = ""

    for tag, attrs in seletores:
        bloco = soup.find(tag, attrs=attrs)
        if bloco:
            texto = limpar_texto(bloco.get_text(" ", strip=True))
            if len(texto) > len(melhor_texto):
                melhor_bloco = bloco
                melhor_texto = texto

    if melhor_bloco is None:
        raise ValueError("Não foi possível localizar o corpo do texto no Senado.")

    html_bloco = str(melhor_bloco)
    texto_bruto = limpar_texto(melhor_bloco.get_text(" ", strip=True))
    texto_corpo_limpo = limpar_texto_norma_final(texto_bruto)
    texto_com_cabecalho = enriquecer_texto_com_cabecalho_canonico(texto_corpo_limpo)

    return {
        "fonte": "senado",
        "url": url,
        "html": html_bloco,
        "texto_bruto": texto_bruto,
        "texto_corpo_limpo": texto_corpo_limpo,
        "texto": texto_com_cabecalho,
        "tamanho_texto_bruto": len(texto_bruto),
        "tamanho_texto_corpo_limpo": len(texto_corpo_limpo),
        "tamanho_texto": len(texto_com_cabecalho)
    }

## Consolidação dos metadados com o texto integral

Nesta etapa será montado um único registro final da norma.

Esse registro reunirá:

- a URN;
- os metadados vindos da API do Senado;
- os links oficiais descobertos no LexML;
- o texto integral extraído da Câmara e do Senado;
- a definição de uma fonte preferida para uso principal.

O objetivo é deixar a norma pronta para etapas posteriores de armazenamento, indexação e busca vetorial.

In [15]:
from datetime import datetime, timezone

def montar_registro_norma(urn: str) -> dict:
    """
    Consolida metadados, links oficiais e textos integrais da norma.

    Estratégia desta versão:
    - usa a API do Senado como base de metadados;
    - tenta obter links oficiais via LexML sem deixar falha de rede quebrar tudo;
    - usa fallback do link do Senado vindo da própria API quando necessário;
    - prioriza Câmara e, na falta dela, Senado;
    - registra erros por origem sem perder o restante do pipeline.
    """
    raw_api = baixar_metadados_api_senado_por_urn(urn, VERSAO_API)
    metadados = extrair_metadados_api(raw_api)
    links = extrair_links_lexml(urn, verbose=False)

    # Fallback: se o LexML não entregar link do Senado, tenta aproveitar urlDocumento da API
    if not links.get("senado") and metadados.get("urlDocumento"):
        links["senado"] = metadados["urlDocumento"]
        if isinstance(links.get("candidatos"), dict):
            links["candidatos"]["senado_api"] = metadados["urlDocumento"]

    registro = {
        "urn": urn,
        "coletado_em": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "metadados_api_senado": metadados,
        "links_oficiais": links,
        "fontes_texto": {},
        "texto_preferido": None,
        "fonte_preferida": None
    }

    # Câmara
    if links.get("camara"):
        try:
            registro["fontes_texto"]["camara"] = extrair_texto_camara(links["camara"])
        except Exception as e:
            registro["fontes_texto"]["camara_erro"] = str(e)

    # Senado
    if links.get("senado"):
        try:
            registro["fontes_texto"]["senado"] = extrair_texto_senado(links["senado"])
        except Exception as e:
            registro["fontes_texto"]["senado_erro"] = str(e)

    # Define texto preferido
    if "camara" in registro["fontes_texto"]:
        registro["texto_preferido"] = registro["fontes_texto"]["camara"]["texto"]
        registro["fonte_preferida"] = "camara"
    elif "senado" in registro["fontes_texto"]:
        registro["texto_preferido"] = registro["fontes_texto"]["senado"]["texto"]
        registro["fonte_preferida"] = "senado"

    return registro

## Execução do pipeline completo da norma

Nesta etapa será executado o fluxo completo de consolidação, desde a coleta dos metadados até a escolha do texto preferido.

Se tudo funcionar corretamente, ao final desta célula haverá uma estrutura única contendo os dados centrais da norma jurídica.

In [16]:
registro_norma = montar_registro_norma(URN_ALVO)

## Validação da coleta realizada

Nesta etapa será feita uma conferência simples do resultado obtido.

Serão exibidos:
- URN;
- fonte preferida;
- ementa;
- links principais;
- prévia do texto integral.

Essa validação permite confirmar se a coleta e a unificação da norma ocorreram corretamente.

In [17]:
print("URN:", registro_norma["urn"])
print("Fonte preferida:", registro_norma["fonte_preferida"])
print("Ementa:", registro_norma["metadados_api_senado"]["ementa"])
print("Link Câmara:", registro_norma["links_oficiais"]["camara"])
print("Link Senado:", registro_norma["links_oficiais"]["senado"])

print("\n--- PRÉVIA DO TEXTO ---\n")
print(registro_norma["texto_preferido"][:1500] if registro_norma["texto_preferido"] else "Nenhum texto capturado.")

URN: urn:lex:br:federal:lei:1990-12-11;8112
Fonte preferida: camara
Ementa: Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais
Link Câmara: http://www2.camara.gov.br/legin/fed/lei/1990/lei-8112-11-dezembro-1990-322161-publicacaooriginal-1-pl.html
Link Senado: http://legis.senado.leg.br/norma/549988/publicacao/15722628

--- PRÉVIA DO TEXTO ---

LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990
Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais
O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte lei:

TÍTULO I CAPÍTULO ÚNICO Das Disposições Preliminares
Art. 1º Esta Lei institui o regime jurídico dos servidores públicos civis da União, das autarquias, inclusive as em regime especial, e das fundações públicas federais.
Art. 2º Para os efeitos desta Lei, servidor é a pessoa legalmente investida em cargo público.
Art. 3º Ca

## Salvamento do registro final em JSON (Base)

Após a consolidação dos dados, o resultado será salvo em arquivo JSON.

Esse arquivo servirá como base estruturada para as próximas fases do projeto, incluindo enriquecimento do corpus, criação de chunks e posterior inserção em banco vetorial.

In [18]:
from pathlib import Path
import re

# Salvamento robusto do registro final em JSON

if "registro_norma" not in globals():
    raise RuntimeError("registro_norma não está definido. Execute primeiro o pipeline completo da norma.")

if not registro_norma.get("urn"):
    raise RuntimeError("registro_norma['urn'] está vazio. Não é seguro salvar o arquivo final.")

if not registro_norma.get("texto_preferido"):
    raise RuntimeError(
        "registro_norma['texto_preferido'] está vazio. "
        "Não é seguro salvar o registro final sem o texto integral consolidado."
    )

saida_json_dir = Path("saida_json")
saida_json_dir.mkdir(exist_ok=True)

slug_urn_registro = re.sub(r"[^a-zA-Z0-9]+", "_", registro_norma["urn"]).strip("_").lower()
arquivo_registro_json = saida_json_dir / f"{slug_urn_registro}_registro_norma.json"

with open(arquivo_registro_json, "w", encoding="utf-8") as f:
    json.dump(registro_norma, f, ensure_ascii=False, indent=2)

print("Arquivo salvo com sucesso:")
print(arquivo_registro_json)
print("Fonte preferida salva:", registro_norma["fonte_preferida"])
print("Tamanho do texto salvo:", len(registro_norma["texto_preferido"]))

Arquivo salvo com sucesso:
saida_json/urn_lex_br_federal_lei_1990_12_11_8112_registro_norma.json
Fonte preferida salva: camara
Tamanho do texto salvo: 91206


## Preparação do texto para segmentação por artigos

Para uso em recuperação de informação e modelos vetoriais, será necessário dividir o inteiro teor em partes menores.

Nesta etapa será criada uma função para separar a norma por artigos, mantendo a associação com:
- URN;
- fonte;
- identificador do artigo;
- texto correspondente.

In [19]:
def dividir_em_artigos(texto: str, urn: str, fonte: str) -> list:
    if not texto:
        return []

    padrao = r'(?=Art\.\s*\d+[º°]?)'
    partes = re.split(padrao, texto)

    artigos = []
    for i, parte in enumerate(partes):
        parte = parte.strip()
        if not parte:
            continue

        match = re.match(r'Art\.\s*(\d+[º°]?)', parte)
        artigo_id = match.group(1) if match else f"bloco_{i}"

        artigos.append({
            "urn": urn,
            "fonte": fonte,
            "artigo": artigo_id,
            "texto": parte
        })

    return artigos

## Geração e salvamento da segmentação preliminar por artigos

Nesta etapa será gerada uma segmentação preliminar do texto integral da norma, com objetivo de conferência estrutural.

Essa etapa não substitui a geração robusta de chunks jurídicos realizada nas fases posteriores do notebook.

Seu papel aqui é apenas:
- confirmar que o texto consolidado pode ser dividido em blocos por artigo;
- gerar uma visão preliminar da segmentação;
- apoiar a validação do inteiro teor antes da estruturação canônica definitiva.

In [20]:
# Geração e salvamento da segmentação preliminar por artigos

if "registro_norma" not in globals():
    raise RuntimeError("registro_norma não está definido. Execute primeiro o pipeline completo da norma.")

if not registro_norma.get("texto_preferido"):
    raise RuntimeError(
        "registro_norma['texto_preferido'] está vazio. "
        "Não é possível gerar a segmentação preliminar sem o texto integral."
    )

segmentacao_preliminar_artigos = dividir_em_artigos(
    texto=registro_norma["texto_preferido"],
    urn=registro_norma["urn"],
    fonte=registro_norma["fonte_preferida"]
)

print(f"Quantidade de blocos preliminares: {len(segmentacao_preliminar_artigos)}")

saida_json_dir = Path("saida_json")
saida_json_dir.mkdir(exist_ok=True)

arquivo_segmentacao_preliminar = saida_json_dir / "segmentacao_preliminar_artigos.json"

with open(arquivo_segmentacao_preliminar, "w", encoding="utf-8") as f:
    json.dump(segmentacao_preliminar_artigos, f, ensure_ascii=False, indent=2)

print("Arquivo salvo com sucesso:")
print(arquivo_segmentacao_preliminar)

if segmentacao_preliminar_artigos:
    print("\nPrimeiro bloco preliminar:\n")
    print(json.dumps(segmentacao_preliminar_artigos[0], ensure_ascii=False, indent=2)[:2000])

Quantidade de blocos preliminares: 254
Arquivo salvo com sucesso:
saida_json/segmentacao_preliminar_artigos.json

Primeiro bloco preliminar:

{
  "urn": "urn:lex:br:federal:lei:1990-12-11;8112",
  "fonte": "camara",
  "artigo": "bloco_0",
  "texto": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990\nDispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais\nO PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte lei:\n\nTÍTULO I CAPÍTULO ÚNICO Das Disposições Preliminares"
}


## Verificação complementar do texto integral consolidado

Esta etapa realiza uma conferência adicional do texto integral já consolidado, com foco em:

- identificação da norma;
- fonte preferida selecionada;
- ementa oficial;
- links principais das fontes;
- tamanho total do texto capturado;
- visualização do início do conteúdo;
- visualização do fim do conteúdo.

Essa verificação serve como checkpoint antes da evolução para a camada canônica e vetorial das fases seguintes.

In [21]:
print("URN:", registro_norma["urn"])
print("Fonte preferida:", registro_norma["fonte_preferida"])
print("Ementa:", registro_norma["metadados_api_senado"]["ementa"])
print("Link Câmara:", registro_norma["links_oficiais"]["camara"])
print("Link Senado:", registro_norma["links_oficiais"]["senado"])
print("Tamanho total do texto:", len(registro_norma["texto_preferido"]))

print("\n--- INÍCIO ---\n")
print(registro_norma["texto_preferido"][:1800])

print("\n--- FIM ---\n")
print(registro_norma["texto_preferido"][-1200:])

URN: urn:lex:br:federal:lei:1990-12-11;8112
Fonte preferida: camara
Ementa: Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais
Link Câmara: http://www2.camara.gov.br/legin/fed/lei/1990/lei-8112-11-dezembro-1990-322161-publicacaooriginal-1-pl.html
Link Senado: http://legis.senado.leg.br/norma/549988/publicacao/15722628
Tamanho total do texto: 91206

--- INÍCIO ---

LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990
Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais
O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte lei:

TÍTULO I CAPÍTULO ÚNICO Das Disposições Preliminares
Art. 1º Esta Lei institui o regime jurídico dos servidores públicos civis da União, das autarquias, inclusive as em regime especial, e das fundações públicas federais.
Art. 2º Para os efeitos desta Lei, servidor é a pessoa legalmente investida em carg

## Fase 2 - Planejamento do schema JSON para preparação do banco vetorial

Nesta fase o objetivo não é apenas armazenar o inteiro teor da norma, mas estruturar os dados de forma adequada para recuperação semântica, filtragem por metadados e futura indexação em ChromaDB.

A ideia central será separar o problema em duas camadas:

1. uma camada canônica, que preserva a norma completa e seus metadados oficiais;
2. uma camada de chunks, pensada especificamente para embeddings e recuperação vetorial.

Essa separação é importante para manter ao mesmo tempo:
- fidelidade jurídica;
- rastreabilidade da origem do texto;
- flexibilidade para busca semântica e filtros posteriores.

##  Modelagem do schema JSON para preparação do banco vetorial

Nesta fase o objetivo é estruturar a norma jurídica em formatos adequados para duas finalidades distintas:

1. preservar a norma completa com rastreabilidade e consistência entre fontes;
2. preparar unidades menores de texto para recuperação semântica em banco vetorial.

A estratégia adotada será dividir a solução em duas camadas:

- uma camada canônica, responsável por armazenar a norma completa e seus metadados oficiais;
- uma camada vetorial, responsável por armazenar os chunks jurídicos preparados para embeddings e busca semântica.

Essa separação permite manter fidelidade jurídica, rastreabilidade da origem do dado e flexibilidade para futuras consultas em ChromaDB.

## Camada canônica da norma

A camada canônica será o registro mestre de cada norma.

Ela armazenará:
- identificadores;
- metadados oficiais;
- redundâncias observadas entre fontes;
- links oficiais;
- texto integral nas versões relevantes;
- informações de proveniência e auditoria.

Essa camada não será pensada como unidade principal de recuperação vetorial, mas sim como base canônica e rastreável do projeto.

## Camada vetorial da norma

A camada vetorial será composta por chunks jurídicos menores, preparados para embeddings e recuperação semântica.

A unidade prioritária será o artigo, com subdivisão apenas quando necessário.

Cada chunk manterá vínculo com a norma de origem e carregará metadados suficientes para filtragem, rastreabilidade e interpretação posterior dos resultados.

## Mapeamento do schema para o ChromaDB

O banco vetorial será usado para armazenar apenas os chunks jurídicos preparados para recuperação semântica.

A norma completa permanecerá fora da collection principal, em formato canônico e rastreável.

No ChromaDB, cada registro deverá conter:

- um ID estável;
- um documento textual contextualizado;
- metadados planos e filtráveis;
- embedding automático ou embedding previamente calculado, conforme a estratégia adotada.

## Fase 3 - Geração dos arquivos JSON reais da norma

Nesta fase a estrutura consolidada da norma jurídica será transformada em arquivos efetivos para uso posterior no banco vetorial.

Serão gerados dois produtos principais:

1. um JSON canônico, contendo a norma completa, seus metadados, a rastreabilidade entre fontes e as versões textuais relevantes;
2. um conjunto de chunks jurídicos estruturados, preparados para embeddings, recuperação semântica e futura inserção no ChromaDB.

A partir desta fase, o notebook deixa de trabalhar apenas com estruturas em memória e passa a produzir arquivos concretos e reaproveitáveis.

In [22]:
from pathlib import Path
import unicodedata
import hashlib

saida_dir = Path("saida_json")
saida_dir.mkdir(exist_ok=True)

## Funções auxiliares para construção do JSON canônico

Nesta etapa serão definidas funções auxiliares para:

- normalizar valores antes de comparar redundâncias;
- calcular o status de concordância entre fontes;
- extrair observações de título e ementa nas fontes textuais;
- consultar novamente o LexML para enriquecer a rastreabilidade;
- montar o JSON canônico final da norma.

In [23]:
def normalizar_para_comparacao(valor):
    if valor is None:
        return None

    valor = str(valor).strip().lower()

    # remove acentos
    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(c for c in valor if not unicodedata.combining(c))

    # normaliza espaços
    valor = re.sub(r"\s+", " ", valor)

    # normaliza formas comuns de "nº"
    valor = valor.replace("n°", "nº").replace("no.", "nº").replace("n.", "nº")

    # remove pontuação que atrapalha comparação semântica simples
    valor = valor.replace(".", "")
    valor = valor.replace(",", "")
    valor = valor.replace(";", "")
    valor = valor.replace(":", "")

    # remove ponto final sobrando
    valor = valor.strip(" .")

    return valor.strip()


def filtrar_valores_preenchidos(d):
    return {k: v for k, v in d.items() if v not in [None, "", [], {}]}


def calcular_agreement_status(values_by_source: dict) -> str:
    valores = [v for v in values_by_source.values() if v not in [None, ""]]

    if not valores:
        return "missing_in_some_sources"

    valores_norm = [normalizar_para_comparacao(v) for v in valores]

    if len(set(valores)) == 1 and len(valores) == len(values_by_source):
        return "all_equal"

    if len(set(valores_norm)) == 1:
        if len(valores) == len(values_by_source):
            return "normalized_equal"
        return "missing_in_some_sources"

    if len(valores) < len(values_by_source):
        return "missing_in_some_sources"

    return "conflict"

## Geração do JSON canônico da norma

Nesta etapa será montado o JSON canônico efetivo da norma jurídica.

Esse arquivo terá como objetivo preservar:
- a identificação oficial da norma;
- os metadados escolhidos como canônicos;
- a rastreabilidade das redundâncias observadas entre fontes;
- o texto consolidado da norma;
- as informações de pipeline e auditoria.

Esse JSON será a base canônica do projeto e não deve ser confundido com a estrutura que será enviada diretamente ao banco vetorial.

In [24]:
from datetime import datetime, timezone

def primeiro_preenchido(*valores):
    for valor in valores:
        if valor not in [None, "", [], {}]:
            return valor
    return None


def extrair_numero_e_ano_da_urn(urn: str):
    match = re.search(r"lei:(\d{4})-\d{2}-\d{2};([^:]+)$", urn)
    if not match:
        return None, None
    return match.group(2), match.group(1)


def extrair_texto_bruto_fonte(obj):
    if obj is None:
        return None

    if isinstance(obj, str):
        return obj

    if isinstance(obj, dict):
        for chave in [
            "texto",
            "texto_preferido",
            "texto_para_embedding",
            "texto_limpo",
            "texto_final",
            "conteudo",
            "conteudo_limpo",
            "texto_integral",
        ]:
            valor = obj.get(chave)
            if isinstance(valor, str) and valor.strip():
                return valor

    return None


def limpar_linhas_texto(texto):
    if not texto or not isinstance(texto, str):
        return []
    linhas = [linha.strip() for linha in texto.splitlines()]
    return [linha for linha in linhas if linha]


def extrair_titulo_ementa_do_texto(texto):
    linhas = limpar_linhas_texto(texto)

    ignorar = {
        "[Detalhes da Norma]",
        "Este texto não substitui o original publicado no Diário Oficial."
    }
    linhas = [l for l in linhas if l not in ignorar]

    titulo = None
    ementa = None

    for i, linha in enumerate(linhas):
        if re.match(r"^LEI\s+N[º°]?\s*", linha, flags=re.IGNORECASE):
            titulo = linha.strip()

            for prox in linhas[i + 1:i + 6]:
                if not re.match(
                    r"^(O\s+PRESIDENTE\s+DA\s+REP[ÚU]BLICA|TÍTULO\s+[IVXLCDM]+|CAPÍTULO\s+[IVXLCDM]+|Art\.)",
                    prox,
                    flags=re.IGNORECASE
                ):
                    ementa = prox.strip()
                    break
            break

    return titulo, ementa


def montar_json_canonico(registro_norma: dict, urn_alvo: str) -> dict:
    numero_urn, ano_urn = extrair_numero_e_ano_da_urn(urn_alvo)

    metadados_api = registro_norma.get("metadados_api_senado", {})
    links_oficiais = registro_norma.get("links_oficiais", {})

    fontes_texto = registro_norma.get("fontes_texto", {})
    texto_camara = extrair_texto_bruto_fonte(fontes_texto.get("camara"))
    texto_senado = extrair_texto_bruto_fonte(fontes_texto.get("senado"))

    texto_preferido = extrair_texto_bruto_fonte(registro_norma.get("texto_preferido"))
    fonte_preferida = registro_norma.get("fonte_preferida") or "camara"

    if not texto_preferido:
        if fonte_preferida == "camara":
            texto_preferido = primeiro_preenchido(texto_camara, texto_senado)
        elif fonte_preferida == "senado":
            texto_preferido = primeiro_preenchido(texto_senado, texto_camara)
        else:
            texto_preferido = primeiro_preenchido(texto_camara, texto_senado)

    titulo_texto_preferido, ementa_texto_preferido = extrair_titulo_ementa_do_texto(texto_preferido)
    titulo_camara, ementa_camara = extrair_titulo_ementa_do_texto(texto_camara)
    titulo_senado, ementa_senado = extrair_titulo_ementa_do_texto(texto_senado)

    titulo_api = primeiro_preenchido(
        metadados_api.get("titulo_norma"),
        metadados_api.get("titulo")
    )

    ementa_api = primeiro_preenchido(
        metadados_api.get("ementa")
    )

    titulo_lexml = primeiro_preenchido(registro_norma.get("titulo_lexml"))
    ementa_lexml = primeiro_preenchido(registro_norma.get("ementa_lexml"))

    numero = primeiro_preenchido(
        metadados_api.get("numero"),
        registro_norma.get("numero"),
        numero_urn
    )

    ano = primeiro_preenchido(
        metadados_api.get("ano"),
        registro_norma.get("ano"),
        ano_urn
    )

    values_titulo = {
        "api_senado": titulo_api,
        "lexml": titulo_lexml,
        "texto_preferido": titulo_texto_preferido,
        "camara_texto": titulo_camara,
        "senado_texto": titulo_senado
    }

    values_ementa = {
        "api_senado": ementa_api,
        "lexml": ementa_lexml,
        "texto_preferido": ementa_texto_preferido,
        "camara_texto": ementa_camara,
        "senado_texto": ementa_senado
    }

    titulo_canonico = primeiro_preenchido(
        titulo_api,
        titulo_lexml,
        titulo_texto_preferido,
        titulo_camara,
        titulo_senado
    )

    ementa_canonica = primeiro_preenchido(
        ementa_api,
        ementa_lexml,
        ementa_texto_preferido,
        ementa_camara,
        ementa_senado
    )

    if not titulo_canonico and numero:
        titulo_canonico = f"LEI Nº {numero}"

    return {
        "urn": urn_alvo,
        "canonical": {
            "titulo_norma": titulo_canonico,
            "ementa": ementa_canonica,
            "numero": str(numero) if numero is not None else None,
            "ano": str(ano) if ano is not None else None
        },
        "provenance": {
            "titulo_norma": {
                "chosen_from": (
                    "api_senado" if titulo_api else
                    "lexml" if titulo_lexml else
                    "texto_preferido" if titulo_texto_preferido else
                    "camara_texto" if titulo_camara else
                    "senado_texto"
                ),
                "agreement_status": calcular_agreement_status(values_titulo),
                "values_by_source": values_titulo
            },
            "ementa": {
                "chosen_from": (
                    "api_senado" if ementa_api else
                    "lexml" if ementa_lexml else
                    "texto_preferido" if ementa_texto_preferido else
                    "camara_texto" if ementa_camara else
                    "senado_texto"
                ),
                "agreement_status": calcular_agreement_status(values_ementa),
                "values_by_source": values_ementa
            }
        },
        "links_oficiais": links_oficiais,
        "texto": {
            "fonte_preferida": fonte_preferida,
            "texto_para_embedding": texto_preferido,
            "texto_camara": texto_camara,
            "texto_senado": texto_senado
        },
        "pipeline": {
            "coletado_em": registro_norma.get("coletado_em"),
            "versao_pipeline": "v1"
        }
    }

In [25]:
json_canonico = montar_json_canonico(registro_norma, URN_ALVO)

print("URN:", json_canonico["urn"])
print("Título canônico:", json_canonico["canonical"]["titulo_norma"])
print("Ementa canônica:", json_canonico["canonical"]["ementa"])
print("Status título:", json_canonico["provenance"]["titulo_norma"]["agreement_status"])
print("Status ementa:", json_canonico["provenance"]["ementa"]["agreement_status"])

URN: urn:lex:br:federal:lei:1990-12-11;8112
Título canônico: LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990
Ementa canônica: Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais
Status título: missing_in_some_sources
Status ementa: missing_in_some_sources


In [26]:
arquivo_json_canonico = saida_dir / "norma_canonica_8112.json"

with open(arquivo_json_canonico, "w", encoding="utf-8") as f:
    json.dump(json_canonico, f, ensure_ascii=False, indent=2)

print(f"Arquivo salvo: {arquivo_json_canonico}")

Arquivo salvo: saida_json/norma_canonica_8112.json


## Geração dos chunks jurídicos da norma

Nesta etapa a norma canônica será transformada em unidades menores para uso em recuperação semântica.

A estratégia adotada será priorizar o artigo como unidade principal de chunking, preservando:
- a referência à norma de origem;
- a hierarquia normativa;
- o texto do artigo;
- um texto contextualizado para futura vetorização.

Esses chunks serão a base real para embeddings e inserção no ChromaDB.

In [27]:
def slugificar_urn(urn: str) -> str:
    slug = urn.lower()
    slug = unicodedata.normalize("NFKD", slug)
    slug = "".join(c for c in slug if not unicodedata.combining(c))
    slug = re.sub(r"[^a-z0-9]+", "_", slug)
    slug = re.sub(r"_+", "_", slug).strip("_")
    return slug


def slugificar_artigo(artigo: str | None) -> str | None:
    if not artigo:
        return None

    artigo = str(artigo).strip()
    artigo = unicodedata.normalize("NFKD", artigo)
    artigo = "".join(c for c in artigo if not unicodedata.combining(c))
    artigo = artigo.replace("º", "").replace("°", "")
    artigo = artigo.replace("–", "-")
    artigo = artigo.lower()
    artigo = re.sub(r"[^0-9a-z\-]+", "_", artigo)
    artigo = re.sub(r"_+", "_", artigo).strip("_")
    artigo = artigo.replace("-", "_")
    return artigo


def extrair_linhas_normativas(texto: str) -> list[str]:
    return [linha.strip() for linha in texto.splitlines() if linha.strip()]


def gerar_chunk_id(urn: str, artigo: str | None, ordem_chunk: int) -> str:
    base = slugificar_urn(urn)

    if artigo:
        artigo_slug = slugificar_artigo(artigo)
        return f"{base}__art_{artigo_slug}__chunk_{ordem_chunk}"

    return f"{base}__chunk_{ordem_chunk}"


def montar_texto_contextualizado(titulo_norma, ementa, artigo, texto_chunk):
    partes = []

    if titulo_norma:
        partes.append(titulo_norma)

    if ementa:
        partes.append(ementa)

    if artigo:
        partes.append(f"Art. {artigo}")

    partes.append(texto_chunk)

    return " ".join(partes).strip()


def extrair_identificador_artigo(linha: str) -> str | None:
    """
    Captura somente cabeçalhos reais de artigo.
    Observação importante:
    - é propositalmente case-sensitive em 'Art.'
    - não captura 'art.' em minúsculo dentro de referências internas
    """
    match = re.match(
        r'^Art\.\s*([0-9]+(?:[-–][A-Z]+)?[º°]?)',
        linha
    )
    if not match:
        return None

    artigo = match.group(1)
    artigo = artigo.replace("º", "").replace("°", "")
    artigo = artigo.replace("–", "-")
    return artigo


def gerar_chunks_artigos(json_canonico: dict) -> list[dict]:
    """
    Gera chunks jurídicos com prioridade por artigo.
    Também cria um chunk inicial de cabeçalho/preâmbulo, se houver conteúdo antes do Art. 1º.

    Correções desta versão:
    - ao encontrar novo TÍTULO, zera CAPÍTULO, SEÇÃO e SUBSEÇÃO;
    - ao encontrar novo CAPÍTULO, zera SEÇÃO e SUBSEÇÃO;
    - ao encontrar nova SEÇÃO, zera SUBSEÇÃO;
    - a hierarquia do artigo é congelada no momento em que o artigo começa,
      evitando vazamento de metadados de blocos posteriores.
    """
    urn = json_canonico["urn"]
    titulo_norma = json_canonico["canonical"]["titulo_norma"]
    ementa = json_canonico["canonical"]["ementa"]
    numero = json_canonico["canonical"]["numero"]
    ano = json_canonico["canonical"]["ano"]
    fonte = json_canonico["texto"]["fonte_preferida"]
    url_origem = json_canonico["links_oficiais"].get(fonte)
    coletado_em = json_canonico["pipeline"]["coletado_em"]
    pipeline_version = json_canonico["pipeline"]["versao_pipeline"]

    texto = json_canonico["texto"]["texto_para_embedding"]
    linhas = extrair_linhas_normativas(texto)

    chunks = []

    # Hierarquia corrente do texto
    hierarquia_titulo = None
    hierarquia_capitulo = None
    hierarquia_secao = None
    hierarquia_subsecao = None

    # Snapshot da hierarquia vigente no início do artigo
    artigo_hierarquia_titulo = None
    artigo_hierarquia_capitulo = None
    artigo_hierarquia_secao = None
    artigo_hierarquia_subsecao = None

    preambulo = []
    artigo_numero = None
    artigo_linhas = []
    ordem_chunk = 0

    def salvar_artigo():
        nonlocal ordem_chunk, artigo_numero, artigo_linhas
        nonlocal artigo_hierarquia_titulo, artigo_hierarquia_capitulo
        nonlocal artigo_hierarquia_secao, artigo_hierarquia_subsecao

        if not artigo_numero or not artigo_linhas:
            return

        ordem_chunk += 1
        texto_chunk = limpar_texto(" ".join(artigo_linhas))

        chunks.append({
            "chunk_id": gerar_chunk_id(urn, artigo_numero, ordem_chunk),
            "urn": urn,
            "numero": numero,
            "ano": ano,
            "titulo_norma": titulo_norma,
            "ementa": ementa,
            "fonte": fonte,
            "ordem_chunk": ordem_chunk,
            "tipo_bloco": "artigo",
            "artigo": artigo_numero,
            "paragrafo": None,
            "inciso": None,
            "hierarquia_titulo": artigo_hierarquia_titulo,
            "hierarquia_capitulo": artigo_hierarquia_capitulo,
            "hierarquia_secao": artigo_hierarquia_secao,
            "hierarquia_subsecao": artigo_hierarquia_subsecao,
            "texto_chunk": texto_chunk,
            "texto_contextualizado": montar_texto_contextualizado(
                titulo_norma, ementa, artigo_numero, texto_chunk
            ),
            "url_origem": url_origem,
            "coletado_em": coletado_em,
            "pipeline_version": pipeline_version
        })

    for linha in linhas:
        # Atualização da hierarquia corrente
        if re.match(r'^TÍTULO\s+[IVXLCDM]+', linha):
            hierarquia_titulo = linha
            hierarquia_capitulo = None
            hierarquia_secao = None
            hierarquia_subsecao = None
            continue

        if re.match(r'^CAPÍTULO\s+[IVXLCDM]+', linha):
            hierarquia_capitulo = linha
            hierarquia_secao = None
            hierarquia_subsecao = None
            continue

        if re.match(r'^Seção\s+[IVXLCDM]+', linha):
            hierarquia_secao = linha
            hierarquia_subsecao = None
            continue

        if re.match(r'^Subseção\s+[IVXLCDM]+', linha):
            hierarquia_subsecao = linha
            continue

        # Detecção do início de um novo artigo
        artigo_detectado = extrair_identificador_artigo(linha)
        if artigo_detectado:
            if artigo_linhas:
                salvar_artigo()

            artigo_numero = artigo_detectado
            artigo_linhas = [linha]

            # Congela a hierarquia vigente no momento em que o artigo começa
            artigo_hierarquia_titulo = hierarquia_titulo
            artigo_hierarquia_capitulo = hierarquia_capitulo
            artigo_hierarquia_secao = hierarquia_secao
            artigo_hierarquia_subsecao = hierarquia_subsecao
            continue

        # Conteúdo antes do primeiro artigo
        if artigo_numero is None:
            preambulo.append(linha)
        else:
            artigo_linhas.append(linha)

    if artigo_linhas:
        salvar_artigo()

    if preambulo:
        texto_preambulo = limpar_texto(" ".join(preambulo))
        if texto_preambulo:
            chunk_preambulo = {
                "chunk_id": gerar_chunk_id(urn, None, 0),
                "urn": urn,
                "numero": numero,
                "ano": ano,
                "titulo_norma": titulo_norma,
                "ementa": ementa,
                "fonte": fonte,
                "ordem_chunk": 0,
                "tipo_bloco": "cabecalho_preambulo",
                "artigo": None,
                "paragrafo": None,
                "inciso": None,
                "hierarquia_titulo": None,
                "hierarquia_capitulo": None,
                "hierarquia_secao": None,
                "hierarquia_subsecao": None,
                "texto_chunk": texto_preambulo,
                "texto_contextualizado": montar_texto_contextualizado(
                    titulo_norma, ementa, None, texto_preambulo
                ),
                "url_origem": url_origem,
                "coletado_em": coletado_em,
                "pipeline_version": pipeline_version
            }
            chunks.insert(0, chunk_preambulo)

    return chunks

## Execução da geração dos chunks jurídicos

Nesta etapa o notebook irá percorrer o texto consolidado da norma, identificar os artigos e gerar uma estrutura de chunks com metadados próprios.

O resultado será usado como base direta para a futura criação da collection vetorial no ChromaDB.

In [28]:
chunks_norma = gerar_chunks_artigos(json_canonico)

print("Quantidade total de chunks:", len(chunks_norma))
print("\nPrimeiro chunk:\n")
print(json.dumps(chunks_norma[0], ensure_ascii=False, indent=2)[:2000])

Quantidade total de chunks: 254

Primeiro chunk:

{
  "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__chunk_0",
  "urn": "urn:lex:br:federal:lei:1990-12-11;8112",
  "numero": "8112",
  "ano": "1990",
  "titulo_norma": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990",
  "ementa": "Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais",
  "fonte": "camara",
  "ordem_chunk": 0,
  "tipo_bloco": "cabecalho_preambulo",
  "artigo": null,
  "paragrafo": null,
  "inciso": null,
  "hierarquia_titulo": null,
  "hierarquia_capitulo": null,
  "hierarquia_secao": null,
  "hierarquia_subsecao": null,
  "texto_chunk": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990 Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte lei:",
  "texto_contextualizado": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 199

In [29]:
arquivo_chunks_json = saida_dir / "norma_chunks_8112.json"
arquivo_chunks_jsonl = saida_dir / "norma_chunks_8112.jsonl"

with open(arquivo_chunks_json, "w", encoding="utf-8") as f:
    json.dump(chunks_norma, f, ensure_ascii=False, indent=2)

with open(arquivo_chunks_jsonl, "w", encoding="utf-8") as f:
    for chunk in chunks_norma:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Arquivo salvo: {arquivo_chunks_json}")
print(f"Arquivo salvo: {arquivo_chunks_jsonl}")

Arquivo salvo: saida_json/norma_chunks_8112.json
Arquivo salvo: saida_json/norma_chunks_8112.jsonl


In [30]:
print("JSON canônico salvo em:", arquivo_json_canonico)
print("Chunks JSON salvos em:", arquivo_chunks_json)
print("Chunks JSONL salvos em:", arquivo_chunks_jsonl)

print("\nResumo:")
print("- título canônico:", json_canonico["canonical"]["titulo_norma"])
print("- fonte preferida:", json_canonico["texto"]["fonte_preferida"])
print("- quantidade de chunks:", len(chunks_norma))
print("- tipos de chunk:", sorted(set(chunk["tipo_bloco"] for chunk in chunks_norma)))

JSON canônico salvo em: saida_json/norma_canonica_8112.json
Chunks JSON salvos em: saida_json/norma_chunks_8112.json
Chunks JSONL salvos em: saida_json/norma_chunks_8112.jsonl

Resumo:
- título canônico: LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990
- fonte preferida: camara
- quantidade de chunks: 254
- tipos de chunk: ['artigo', 'cabecalho_preambulo']


## Testes de validação dos chunks jurídicos

Nesta etapa são executadas verificações estruturais sobre os chunks jurídicos gerados a partir do JSON canônico da norma.

O objetivo é confirmar que:

- a quantidade total de chunks está consistente;
- o bloco inicial de cabeçalho/preâmbulo foi preservado corretamente;
- os artigos jurídicos foram segmentados sem duplicações indevidas;
- a estrutura final está pronta para indexação vetorial e recuperação de contexto.

Esses testes representam a validação da camada oficial de chunks do projeto, diferindo da segmentação preliminar usada apenas como conferência nas etapas anteriores.

In [31]:
chunks_norma = gerar_chunks_artigos(json_canonico)

print("Quantidade total de chunks:", len(chunks_norma))
print("Primeiro tipo_bloco:", chunks_norma[0]["tipo_bloco"])
print("Primeiro chunk_id:", chunks_norma[0]["chunk_id"])
print("Primeiro artigo:", chunks_norma[0]["artigo"])

Quantidade total de chunks: 254
Primeiro tipo_bloco: cabecalho_preambulo
Primeiro chunk_id: urn_lex_br_federal_lei_1990_12_11_8112__chunk_0
Primeiro artigo: None


In [32]:
artigos = [c["artigo"] for c in chunks_norma if c["tipo_bloco"] == "artigo" and c["artigo"] is not None]

from collections import Counter
contagem = Counter(artigos)
duplicados = {k: v for k, v in contagem.items() if v > 1}

print("Quantidade de chunks do tipo artigo:", len(artigos))
print("Quantidade de artigos únicos:", len(set(artigos)))
print("Quantidade de artigos duplicados:", len(duplicados))
print("Alguns duplicados:", list(duplicados.items())[:10])

Quantidade de chunks do tipo artigo: 253
Quantidade de artigos únicos: 253
Quantidade de artigos duplicados: 0
Alguns duplicados: []


In [33]:
campos_obrigatorios = [
    "chunk_id", "urn", "numero", "ano", "titulo_norma", "ementa",
    "fonte", "ordem_chunk", "tipo_bloco", "artigo",
    "texto_chunk", "texto_contextualizado"
]

problemas = []

for i, chunk in enumerate(chunks_norma):
    for campo in campos_obrigatorios:
        if campo not in chunk:
            problemas.append((i, f"campo ausente: {campo}"))

    if not chunk.get("chunk_id"):
        problemas.append((i, "chunk_id vazio"))

    if chunk.get("tipo_bloco") == "artigo":
        if not chunk.get("texto_chunk"):
            problemas.append((i, "texto_chunk vazio em artigo"))
        if not chunk.get("texto_contextualizado"):
            problemas.append((i, "texto_contextualizado vazio em artigo"))

print("Quantidade de problemas encontrados:", len(problemas))
print("Amostra de problemas:", problemas[:10])

Quantidade de problemas encontrados: 0
Amostra de problemas: []


In [34]:
artigos_teste = ["1", "5", "37", "116", "243"]

for art in artigos_teste:
    encontrados = [c for c in chunks_norma if c["artigo"] == art]
    print(f"\n=== Artigo {art} ===")
    print("Quantidade encontrada:", len(encontrados))
    if encontrados:
        print("chunk_id:", encontrados[0]["chunk_id"])
        print("tipo_bloco:", encontrados[0]["tipo_bloco"])
        print("texto_chunk:", encontrados[0]["texto_chunk"][:500])


=== Artigo 1 ===
Quantidade encontrada: 1
chunk_id: urn_lex_br_federal_lei_1990_12_11_8112__art_1__chunk_1
tipo_bloco: artigo
texto_chunk: Art. 1º Esta Lei institui o regime jurídico dos servidores públicos civis da União, das autarquias, inclusive as em regime especial, e das fundações públicas federais.

=== Artigo 5 ===
Quantidade encontrada: 1
chunk_id: urn_lex_br_federal_lei_1990_12_11_8112__art_5__chunk_5
tipo_bloco: artigo
texto_chunk: Art. 5º São requisitos básicos para investidura em cargo público: I - a nacionalidade brasileira; II - o gozo dos direitos políticos; III - a quitação com as obrigações militares e eleitorais; IV - o nível de escolaridade exigido para o exercício do cargo; V - a idade mínima de dezoito anos; VI - aptidão física e mental. § 1º As atribuições do cargo podem justificar a exigência de outros requisitos estabelecidos em lei. § 2º Às pessoas portadoras de deficiência é assegurado o direito de se inscr

=== Artigo 37 ===
Quantidade encontrada: 1
chunk_i

In [35]:
arquivo_chunks_json = saida_dir / "norma_chunks_8112.json"
arquivo_chunks_jsonl = saida_dir / "norma_chunks_8112.jsonl"

with open(arquivo_chunks_json, "w", encoding="utf-8") as f:
    json.dump(chunks_norma, f, ensure_ascii=False, indent=2)

with open(arquivo_chunks_jsonl, "w", encoding="utf-8") as f:
    for chunk in chunks_norma:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("Arquivo salvo:", arquivo_chunks_json)
print("Arquivo salvo:", arquivo_chunks_jsonl)

Arquivo salvo: saida_json/norma_chunks_8112.json
Arquivo salvo: saida_json/norma_chunks_8112.jsonl


## Fase 5.1 — Configuração do PostgreSQL com pgvector

Nesta fase iniciaremos a nova infraestrutura de armazenamento vetorial do projeto, substituindo o banco vetorial anterior por uma base em **PostgreSQL** com a extensão **pgvector**.

### Objetivo técnico
Preparar um ambiente open-source e gratuito para armazenar:
- os chunks jurídicos da norma;
- os metadados de cada chunk;
- e, nas próximas fases, os vetores (embeddings) usados na busca semântica.

### Decisão técnica adotada
Foi escolhido o PostgreSQL porque:
- é open-source;
- permite uso relacional e vetorial no mesmo banco;
- já possui extensão vetorial madura (`pgvector`);
- o orientador do projeto já possui familiaridade com essa tecnologia.

### Onde o PostgreSQL será executado nesta etapa
Nesta fase, o PostgreSQL será criado e executado **no próprio ambiente do Colab**, apenas para desenvolvimento e validação da arquitetura.

Isso significa que:
- o banco funciona localmente dentro da sessão atual do notebook;
- ele não é persistente entre reinicializações de sessão;
- a estrutura poderá ser migrada posteriormente para um PostgreSQL persistente.

### Observação metodológica
Nesta fase ainda **não será definida a métrica vetorial final** (por exemplo: cosine distance, inner product ou L2).
Essa decisão matemática será documentada na fase de geração dos embeddings e consulta vetorial.

In [36]:
# Fase 5.1 - Instalação e preparação do PostgreSQL com pgvector
#
# Esta célula executa a preparação do ambiente local da sessão:
# 1. instala PostgreSQL e dependências de compilação;
# 2. instala a biblioteca Python de conexão (psycopg2);
# 3. inicia o serviço do PostgreSQL;
# 4. baixa e compila a extensão pgvector;
# 5. cria o banco do projeto;
# 6. habilita a extensão vetorial no banco.
#
# Importante:
# - esta estrutura existe apenas durante a sessão atual do Colab;
# - não há chamadas externas de inferência nesta fase.

!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib postgresql-server-dev-all build-essential git
!pip install -q psycopg2-binary

# Inicia o PostgreSQL local da sessão atual
!service postgresql start

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package logrotate.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../00-logrotate_3.19.0-1ubuntu1.1_amd64.deb ...
Unpacking logrotate (3.19.0-1ubuntu1.1) ...
Selecting previously unselected package netbase.
Preparing to unpack .../01-netbase_6.3_all.deb ...
Unpacking netbase (6.3) ...
Selecting previously unselected package python3-yaml.
Preparing to unpack .../02-python3-yaml_5.4.1-1ubuntu1_amd64.deb ...
Unpacking python3-yaml (5.4.1-1ubuntu1) ...
Selecting previously unselected package binfmt-support.
Preparing to unpack .../03-binfmt-support_2.2.1-2_amd64.deb ...
Unpacking binfmt-support (2.2.1-2) ...
Selecting previously unselected package libllvm14:a

In [37]:
# Fase 5.1 - Instalação da extensão pgvector
#
# A extensão pgvector permite criar colunas do tipo vetorial no PostgreSQL.
# Nesta etapa, ela é compilada diretamente no ambiente do notebook.
#
# Estratégia:
# - se o diretório /content/pgvector ainda não existir, ele é clonado do repositório oficial;
# - em seguida, a extensão é compilada e instalada localmente.

!test -d /content/pgvector || git clone --depth 1 https://github.com/pgvector/pgvector.git /content/pgvector
!cd /content/pgvector && make && make install

Cloning into '/content/pgvector'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 163 (delta 85), reused 54 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 150.96 KiB | 7.19 MiB/s, done.
Resolving deltas: 100% (85/85), done.
gcc -Wall -Wmissing-prototypes -Wpointer-arith -Wdeclaration-after-statement -Werror=vla -Wendif-labels -Wmissing-format-attribute -Wimplicit-fallthrough=3 -Wcast-function-type -Wformat-security -fno-strict-aliasing -fwrapv -fexcess-precision=standard -Wno-format-truncation -Wno-stringop-truncation -g -g -O2 -flto=auto -ffat-lto-objects -flto=auto -ffat-lto-objects -fstack-protector-strong -Wformat -Werror=format-security -fno-omit-frame-pointer -march=native -ftree-vectorize -fassociative-math -fno-signed-zeros -fno-trapping-math -fPIC -I. -I./ -I/usr/include/postgresql/14/server -I/usr/include/postgresql/internal -Wdate-time -D_FORTIFY_

In [38]:
# Fase 5.1 - Criação do banco do projeto e ativação da extensão vector
#
# Banco criado nesta fase:
# - nome: lexml_juridico
# - usuário: postgres
# - host: localhost
# - porta: 5432
#
# A senha do usuário postgres é definida apenas para facilitar a conexão Python
# dentro desta sessão de desenvolvimento no Colab.

!runuser -u postgres -- psql -c "ALTER USER postgres PASSWORD 'postgres';"

# Cria o banco somente se ele ainda não existir
!runuser -u postgres -- psql -tAc "SELECT 1 FROM pg_database WHERE datname='lexml_juridico'" | grep -q 1 || runuser -u postgres -- createdb lexml_juridico

# Ativa a extensão vetorial no banco do projeto
!runuser -u postgres -- psql -d lexml_juridico -c "CREATE EXTENSION IF NOT EXISTS vector;"

ALTER ROLE
CREATE EXTENSION


## Configuração da conexão Python com o PostgreSQL

Nesta etapa centralizamos os parâmetros da conexão Python com o banco local criado no Colab.

Isso é importante porque deixa explícito:
- qual host está sendo usado;
- qual porta está sendo usada;
- qual banco foi criado;
- e com qual usuário a aplicação vai se conectar.

In [39]:
# Fase 5.1 - Configuração Python da conexão com PostgreSQL
#
# Os parâmetros abaixo serão reutilizados nas próximas fases:
# - criação da tabela dos chunks jurídicos;
# - carga dos dados;
# - geração da camada vetorial;
# - consultas semânticas futuras.

import psycopg2
from psycopg2.extras import RealDictCursor

PG_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "lexml_juridico",
    "user": "postgres",
    "password": "postgres"
}

print("Configuração PostgreSQL carregada com sucesso:")
print(PG_CONFIG)

Configuração PostgreSQL carregada com sucesso:
{'host': 'localhost', 'port': 5432, 'dbname': 'lexml_juridico', 'user': 'postgres', 'password': 'postgres'}


In [40]:
# Fase 5.1 - Teste de conexão e verificação da extensão vector
#
# Esta célula confirma tecnicamente:
# 1. se o banco está acessível via Python;
# 2. qual é o banco atual;
# 3. se a extensão pgvector foi ativada corretamente.

def testar_conexao_postgres(config: dict) -> dict:
    with psycopg2.connect(**config) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("SELECT current_database() AS banco_atual;")
            banco = cur.fetchone()

            cur.execute("""
                SELECT extname
                FROM pg_extension
                WHERE extname = 'vector';
            """)
            extensao = cur.fetchone()

    return {
        "banco_atual": banco["banco_atual"],
        "pgvector_ativo": extensao is not None
    }

resultado_teste_pg = testar_conexao_postgres(PG_CONFIG)

print("Resultado do teste PostgreSQL:")
print(resultado_teste_pg)

Resultado do teste PostgreSQL:
{'banco_atual': 'lexml_juridico', 'pgvector_ativo': True}


## Critério de aprovação da Fase 5.1

A fase será considerada aprovada se o teste final retornar:

- `banco_atual = lexml_juridico`
- `pgvector_ativo = True`

Se isso ocorrer, a infraestrutura inicial do PostgreSQL estará pronta e poderemos avançar para a modelagem da tabela dos chunks jurídicos.

## Fase 5.2 — Modelagem da tabela dos chunks jurídicos no PostgreSQL

Nesta fase será criada a primeira estrutura persistente da norma dentro do PostgreSQL.

### Objetivo técnico
Criar uma tabela relacional para armazenar os chunks jurídicos já gerados anteriormente, preservando:

- identificador único do chunk;
- URN da norma;
- tipo de bloco jurídico;
- número do artigo, quando existir;
- texto do chunk;
- ordem do chunk dentro da norma.

### Decisão de modelagem
Nesta etapa, a tabela será criada **sem a coluna vetorial**.

Isso foi feito propositalmente para separar com clareza duas responsabilidades:

1. primeiro, validar a persistência relacional dos chunks;
2. depois, em fase própria, adicionar os embeddings e a busca vetorial.

### Benefício dessa abordagem
Essa separação melhora:
- clareza metodológica;
- facilidade de depuração;
- controle da evolução da arquitetura;
- explicação acadêmica do pipeline.

In [41]:
# Fase 5.2 - Inspeção da estrutura atual dos chunks
#
# Antes de criar a tabela, esta célula valida o formato dos chunks já gerados
# na fase anterior. O objetivo é confirmar quais campos já existem e como eles
# poderão ser mapeados para a estrutura relacional do PostgreSQL.

print("Quantidade total de chunks:", len(chunks_norma))
print("\nChaves do primeiro chunk:")
print(sorted(chunks_norma[0].keys()))

print("\nPrimeiro chunk:")
print(json.dumps(chunks_norma[0], ensure_ascii=False, indent=2)[:3000])

Quantidade total de chunks: 254

Chaves do primeiro chunk:
['ano', 'artigo', 'chunk_id', 'coletado_em', 'ementa', 'fonte', 'hierarquia_capitulo', 'hierarquia_secao', 'hierarquia_subsecao', 'hierarquia_titulo', 'inciso', 'numero', 'ordem_chunk', 'paragrafo', 'pipeline_version', 'texto_chunk', 'texto_contextualizado', 'tipo_bloco', 'titulo_norma', 'url_origem', 'urn']

Primeiro chunk:
{
  "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__chunk_0",
  "urn": "urn:lex:br:federal:lei:1990-12-11;8112",
  "numero": "8112",
  "ano": "1990",
  "titulo_norma": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990",
  "ementa": "Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais",
  "fonte": "camara",
  "ordem_chunk": 0,
  "tipo_bloco": "cabecalho_preambulo",
  "artigo": null,
  "paragrafo": null,
  "inciso": null,
  "hierarquia_titulo": null,
  "hierarquia_capitulo": null,
  "hierarquia_secao": null,
  "hierarquia_subsecao": null,
  "texto

## Esquema relacional inicial dos chunks jurídicos

Com base na estrutura atual dos chunks, será criada uma tabela chamada `chunks_juridicos`.

### Campos planejados nesta fase
- `id` — chave primária sequencial;
- `chunk_id` — identificador textual único do chunk;
- `urn` — identificador da norma;
- `tipo_bloco` — tipo jurídico do bloco;
- `artigo` — número do artigo, quando existir;
- `texto_chunk` — conteúdo textual do chunk;
- `ordem_chunk` — posição do chunk dentro da norma;
- `metadata_json` — metadados completos do chunk em JSONB.

### Observação
A coluna vetorial será adicionada em fase posterior, depois que a carga relacional básica estiver validada.

In [42]:
# Fase 5.2 - Criação da tabela relacional dos chunks jurídicos
#
# Esta tabela servirá como base persistente da norma no PostgreSQL.
# Ainda não há coluna vetorial nesta etapa.
# O foco aqui é garantir integridade estrutural e compatibilidade com a fase
# seguinte de ingestão dos dados.

def criar_tabela_chunks_juridicos(config: dict) -> None:
    sql = """
    DROP TABLE IF EXISTS chunks_juridicos;

    CREATE TABLE chunks_juridicos (
        id SERIAL PRIMARY KEY,
        chunk_id TEXT UNIQUE NOT NULL,
        urn TEXT NOT NULL,
        tipo_bloco TEXT,
        artigo TEXT,
        texto_chunk TEXT NOT NULL,
        ordem_chunk INTEGER NOT NULL,
        metadata_json JSONB NOT NULL
    );
    """

    with psycopg2.connect(**config) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()

criar_tabela_chunks_juridicos(PG_CONFIG)

print("Tabela 'chunks_juridicos' criada com sucesso.")

Tabela 'chunks_juridicos' criada com sucesso.


In [43]:
# Fase 5.2 - Verificação da estrutura da tabela criada
#
# Esta célula consulta o catálogo do PostgreSQL para listar as colunas criadas
# na tabela chunks_juridicos. Isso confirma se o esquema relacional ficou
# conforme o planejado.

def obter_colunas_tabela(config: dict, nome_tabela: str) -> list:
    sql = """
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = %s
    ORDER BY ordinal_position;
    """

    with psycopg2.connect(**config) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(sql, (nome_tabela,))
            return cur.fetchall()

colunas_chunks_juridicos = obter_colunas_tabela(PG_CONFIG, "chunks_juridicos")

print("Colunas da tabela 'chunks_juridicos':")
for coluna in colunas_chunks_juridicos:
    print(f"- {coluna['column_name']} ({coluna['data_type']})")

Colunas da tabela 'chunks_juridicos':
- id (integer)
- chunk_id (text)
- urn (text)
- tipo_bloco (text)
- artigo (text)
- texto_chunk (text)
- ordem_chunk (integer)
- metadata_json (jsonb)


## Critério de aprovação da Fase 5.2

A fase será considerada aprovada se:

- a tabela `chunks_juridicos` for criada sem erro;
- as colunas esperadas forem listadas corretamente;
- a modelagem ficar pronta para a próxima fase de carga dos chunks no PostgreSQL.

## Fase 5.2.A — Refinamento do esquema relacional dos chunks

Antes da carga dos dados no PostgreSQL, será feito um refinamento do esquema relacional da tabela `chunks_juridicos`.

### Motivo do ajuste
Embora os metadados completos já estejam preservados em `metadata_json`, alguns campos são importantes demais para ficarem apenas dentro do JSONB.

### Campos que passarão a existir como colunas explícitas
- `titulo_norma`
- `fonte`
- `texto_contextualizado`
- `url_origem`
- `pipeline_version`
- `hierarquia_titulo`
- `hierarquia_capitulo`
- `hierarquia_secao`
- `hierarquia_subsecao`

### Benefício técnico
Esse refinamento melhora:
- clareza metodológica;
- legibilidade do banco;
- facilidade de auditoria;
- e preparação para a futura camada vetorial.

In [44]:
# Fase 5.2.A - Refinamento da tabela chunks_juridicos
#
# Esta célula recria a tabela com colunas adicionais explícitas.
# A decisão foi tomada para deixar o banco mais transparente tecnicamente
# e mais preparado para as próximas fases de busca vetorial e relatórios.

def recriar_tabela_chunks_juridicos_refinada(config: dict) -> None:
    sql = """
    DROP TABLE IF EXISTS chunks_juridicos;

    CREATE TABLE chunks_juridicos (
        id SERIAL PRIMARY KEY,
        chunk_id TEXT UNIQUE NOT NULL,
        urn TEXT NOT NULL,
        numero TEXT,
        ano TEXT,
        titulo_norma TEXT,
        ementa TEXT,
        fonte TEXT,
        tipo_bloco TEXT,
        artigo TEXT,
        paragrafo TEXT,
        inciso TEXT,
        hierarquia_titulo TEXT,
        hierarquia_capitulo TEXT,
        hierarquia_secao TEXT,
        hierarquia_subsecao TEXT,
        texto_chunk TEXT NOT NULL,
        texto_contextualizado TEXT,
        url_origem TEXT,
        coletado_em TEXT,
        pipeline_version TEXT,
        ordem_chunk INTEGER NOT NULL,
        metadata_json JSONB NOT NULL
    );
    """

    with psycopg2.connect(**config) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()

recriar_tabela_chunks_juridicos_refinada(PG_CONFIG)

print("Tabela 'chunks_juridicos' refinada recriada com sucesso.")

Tabela 'chunks_juridicos' refinada recriada com sucesso.


In [45]:
# Fase 5.2.A - Verificação da estrutura refinada da tabela

colunas_chunks_juridicos_refinada = obter_colunas_tabela(PG_CONFIG, "chunks_juridicos")

print("Colunas da tabela refinada 'chunks_juridicos':")
for coluna in colunas_chunks_juridicos_refinada:
    print(f"- {coluna['column_name']} ({coluna['data_type']})")

Colunas da tabela refinada 'chunks_juridicos':
- id (integer)
- chunk_id (text)
- urn (text)
- numero (text)
- ano (text)
- titulo_norma (text)
- ementa (text)
- fonte (text)
- tipo_bloco (text)
- artigo (text)
- paragrafo (text)
- inciso (text)
- hierarquia_titulo (text)
- hierarquia_capitulo (text)
- hierarquia_secao (text)
- hierarquia_subsecao (text)
- texto_chunk (text)
- texto_contextualizado (text)
- url_origem (text)
- coletado_em (text)
- pipeline_version (text)
- ordem_chunk (integer)
- metadata_json (jsonb)


## Fase 5.3 — Carga dos chunks jurídicos no PostgreSQL

Nesta fase, os chunks jurídicos já gerados anteriormente serão carregados na tabela refinada `chunks_juridicos`.

### Objetivo técnico
Persistir no PostgreSQL a estrutura já validada dos chunks da norma, garantindo:

- integridade dos identificadores;
- preservação da ordem dos chunks;
- armazenamento dos campos explícitos em colunas relacionais;
- armazenamento do metadado completo em `JSONB`.

### Estratégia adotada
A carga será feita por uma função de transformação e uma função de inserção em lote.

Além disso, a inserção será **idempotente**, ou seja:
- se a carga for executada novamente,
- os registros já existentes serão atualizados com base no `chunk_id`,
- evitando duplicidade e mantendo consistência.

### Benefício dessa abordagem
Essa estratégia melhora:
- reprodutibilidade;
- manutenção;
- depuração;
- segurança para reruns do notebook.

In [46]:
# Fase 5.3 - Preparação dos registros para carga no PostgreSQL
#
# Esta célula transforma os chunks jurídicos já existentes em uma lista de
# registros compatíveis com a tabela refinada 'chunks_juridicos'.
#
# Decisão de modelagem:
# - os campos principais são extraídos para colunas explícitas;
# - o chunk completo também é preservado em metadata_json;
# - isso mantém o banco legível e, ao mesmo tempo, auditável.

from psycopg2.extras import Json

def transformar_chunk_em_registro(chunk: dict) -> dict:
    """
    Converte um chunk jurídico em um registro compatível com a tabela
    chunks_juridicos.

    Mantém os campos principais em colunas explícitas e preserva o conteúdo
    integral do chunk em metadata_json para rastreabilidade.
    """
    return {
        "chunk_id": chunk.get("chunk_id"),
        "urn": chunk.get("urn"),
        "numero": chunk.get("numero"),
        "ano": chunk.get("ano"),
        "titulo_norma": chunk.get("titulo_norma"),
        "ementa": chunk.get("ementa"),
        "fonte": chunk.get("fonte"),
        "tipo_bloco": chunk.get("tipo_bloco"),
        "artigo": chunk.get("artigo"),
        "paragrafo": chunk.get("paragrafo"),
        "inciso": chunk.get("inciso"),
        "hierarquia_titulo": chunk.get("hierarquia_titulo"),
        "hierarquia_capitulo": chunk.get("hierarquia_capitulo"),
        "hierarquia_secao": chunk.get("hierarquia_secao"),
        "hierarquia_subsecao": chunk.get("hierarquia_subsecao"),
        "texto_chunk": chunk.get("texto_chunk"),
        "texto_contextualizado": chunk.get("texto_contextualizado"),
        "url_origem": chunk.get("url_origem"),
        "coletado_em": chunk.get("coletado_em"),
        "pipeline_version": chunk.get("pipeline_version"),
        "ordem_chunk": chunk.get("ordem_chunk"),
        "metadata_json": Json(chunk)
    }

registros_chunks_pg = [transformar_chunk_em_registro(chunk) for chunk in chunks_norma]

print("Quantidade de registros preparados para carga:", len(registros_chunks_pg))
print("\nPrévia do primeiro registro preparado:")
preview_registro = registros_chunks_pg[0].copy()
preview_registro["metadata_json"] = "<Json(chunk)>"
print(json.dumps(preview_registro, ensure_ascii=False, indent=2)[:3000])

Quantidade de registros preparados para carga: 254

Prévia do primeiro registro preparado:
{
  "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__chunk_0",
  "urn": "urn:lex:br:federal:lei:1990-12-11;8112",
  "numero": "8112",
  "ano": "1990",
  "titulo_norma": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990",
  "ementa": "Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais",
  "fonte": "camara",
  "tipo_bloco": "cabecalho_preambulo",
  "artigo": null,
  "paragrafo": null,
  "inciso": null,
  "hierarquia_titulo": null,
  "hierarquia_capitulo": null,
  "hierarquia_secao": null,
  "hierarquia_subsecao": null,
  "texto_chunk": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990 Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte lei:",
  "texto_contextualizado": "LEI Nº 8112, DE 

## Inserção em lote com atualização por `chunk_id`

Nesta etapa, os registros serão inseridos em lote no PostgreSQL.

### Estratégia de integridade
Será usado `ON CONFLICT (chunk_id) DO UPDATE`, o que significa que:

- se o `chunk_id` ainda não existir, o registro será inserido;
- se o `chunk_id` já existir, o registro será atualizado.

### Motivo da escolha
Essa abordagem é mais segura para notebooks, porque permite reexecução sem gerar duplicidade.

In [47]:
# Fase 5.3 - Carga em lote dos chunks jurídicos no PostgreSQL
#
# Esta célula realiza a inserção dos registros na tabela chunks_juridicos.
# A carga é idempotente por usar ON CONFLICT (chunk_id) DO UPDATE.
#
# Isso é importante para permitir reruns seguros do notebook sem duplicar dados.

from psycopg2.extras import execute_values

def carregar_chunks_no_postgres(config: dict, registros: list[dict]) -> int:
    """
    Insere ou atualiza em lote os registros dos chunks jurídicos na tabela
    chunks_juridicos.

    Retorna a quantidade de registros processados.
    """
    sql = """
    INSERT INTO chunks_juridicos (
        chunk_id,
        urn,
        numero,
        ano,
        titulo_norma,
        ementa,
        fonte,
        tipo_bloco,
        artigo,
        paragrafo,
        inciso,
        hierarquia_titulo,
        hierarquia_capitulo,
        hierarquia_secao,
        hierarquia_subsecao,
        texto_chunk,
        texto_contextualizado,
        url_origem,
        coletado_em,
        pipeline_version,
        ordem_chunk,
        metadata_json
    )
    VALUES %s
    ON CONFLICT (chunk_id)
    DO UPDATE SET
        urn = EXCLUDED.urn,
        numero = EXCLUDED.numero,
        ano = EXCLUDED.ano,
        titulo_norma = EXCLUDED.titulo_norma,
        ementa = EXCLUDED.ementa,
        fonte = EXCLUDED.fonte,
        tipo_bloco = EXCLUDED.tipo_bloco,
        artigo = EXCLUDED.artigo,
        paragrafo = EXCLUDED.paragrafo,
        inciso = EXCLUDED.inciso,
        hierarquia_titulo = EXCLUDED.hierarquia_titulo,
        hierarquia_capitulo = EXCLUDED.hierarquia_capitulo,
        hierarquia_secao = EXCLUDED.hierarquia_secao,
        hierarquia_subsecao = EXCLUDED.hierarquia_subsecao,
        texto_chunk = EXCLUDED.texto_chunk,
        texto_contextualizado = EXCLUDED.texto_contextualizado,
        url_origem = EXCLUDED.url_origem,
        coletado_em = EXCLUDED.coletado_em,
        pipeline_version = EXCLUDED.pipeline_version,
        ordem_chunk = EXCLUDED.ordem_chunk,
        metadata_json = EXCLUDED.metadata_json;
    """

    valores = [
        (
            r["chunk_id"],
            r["urn"],
            r["numero"],
            r["ano"],
            r["titulo_norma"],
            r["ementa"],
            r["fonte"],
            r["tipo_bloco"],
            r["artigo"],
            r["paragrafo"],
            r["inciso"],
            r["hierarquia_titulo"],
            r["hierarquia_capitulo"],
            r["hierarquia_secao"],
            r["hierarquia_subsecao"],
            r["texto_chunk"],
            r["texto_contextualizado"],
            r["url_origem"],
            r["coletado_em"],
            r["pipeline_version"],
            r["ordem_chunk"],
            r["metadata_json"]
        )
        for r in registros
    ]

    with psycopg2.connect(**config) as conn:
        with conn.cursor() as cur:
            execute_values(cur, sql, valores, page_size=100)
        conn.commit()

    return len(valores)

quantidade_processada_chunks = carregar_chunks_no_postgres(PG_CONFIG, registros_chunks_pg)

print("Carga concluída com sucesso.")
print("Quantidade de registros processados:", quantidade_processada_chunks)

Carga concluída com sucesso.
Quantidade de registros processados: 254


## Fase 5.3.A — Validação da carga dos chunks

Após a inserção, será feita uma validação básica para confirmar:

- quantidade total de registros na tabela;
- consistência do primeiro chunk;
- consistência do último chunk;
- preservação da ordem dos chunks.

Critério de aprovação: a tabela deve conter os 254 registros esperados e a sequência dos chunks deve estar coerente com a estrutura original.

In [48]:
# Fase 5.3.A - Validação da carga dos chunks no PostgreSQL
#
# Esta célula confere se a carga foi realizada corretamente.
# O foco é validar:
# - quantidade total de linhas;
# - primeiro chunk por ordem;
# - último chunk por ordem.

def validar_carga_chunks(config: dict) -> dict:
    with psycopg2.connect(**config) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("SELECT COUNT(*) AS total FROM chunks_juridicos;")
            total = cur.fetchone()["total"]

            cur.execute("""
                SELECT chunk_id, tipo_bloco, artigo, ordem_chunk
                FROM chunks_juridicos
                ORDER BY ordem_chunk ASC
                LIMIT 1;
            """)
            primeiro = cur.fetchone()

            cur.execute("""
                SELECT chunk_id, tipo_bloco, artigo, ordem_chunk
                FROM chunks_juridicos
                ORDER BY ordem_chunk DESC
                LIMIT 1;
            """)
            ultimo = cur.fetchone()

    return {
        "total_registros": total,
        "primeiro_chunk": dict(primeiro) if primeiro else None,
        "ultimo_chunk": dict(ultimo) if ultimo else None
    }

resultado_validacao_carga = validar_carga_chunks(PG_CONFIG)

print("Resultado da validação da carga:")
print(json.dumps(resultado_validacao_carga, ensure_ascii=False, indent=2))

Resultado da validação da carga:
{
  "total_registros": 254,
  "primeiro_chunk": {
    "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__chunk_0",
    "tipo_bloco": "cabecalho_preambulo",
    "artigo": null,
    "ordem_chunk": 0
  },
  "ultimo_chunk": {
    "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__art_253__chunk_253",
    "tipo_bloco": "artigo",
    "artigo": "253",
    "ordem_chunk": 253
  }
}


In [49]:
# Fase 5.3.A - Comparação entre quantidade esperada e quantidade carregada

quantidade_esperada = len(chunks_norma)
quantidade_carga = resultado_validacao_carga["total_registros"]

print("Quantidade esperada:", quantidade_esperada)
print("Quantidade encontrada no PostgreSQL:", quantidade_carga)

if quantidade_carga != quantidade_esperada:
    raise RuntimeError(
        f"A carga ficou inconsistente: esperado={quantidade_esperada}, encontrado={quantidade_carga}"
    )

print("Validação aprovada: a quantidade carregada está correta.")

Quantidade esperada: 254
Quantidade encontrada no PostgreSQL: 254
Validação aprovada: a quantidade carregada está correta.


## Critério de aprovação da Fase 5.3

A fase será considerada aprovada se:

- a carga for concluída sem erro;
- a quantidade de registros no PostgreSQL for igual à quantidade de chunks gerados;
- o primeiro e o último chunk preservarem a ordem esperada.

Se isso ocorrer, a base relacional dos chunks estará pronta para a fase seguinte de embeddings e vetorização.

## Fase 5.4 — Estratégia de embeddings locais e definição da métrica vetorial

Nesta fase será definida a primeira estratégia vetorial do projeto usando uma abordagem totalmente local e open-source.

### Objetivo técnico
Definir:
- o modelo local de embeddings;
- o texto que será transformado em vetor;
- a dimensão esperada do embedding;
- e a métrica vetorial que será usada no PostgreSQL.

### Decisão técnica inicial
Nesta etapa, será utilizada uma arquitetura de embeddings locais via `sentence-transformers`, sem uso de API externa paga.

### Modelo escolhido nesta fase
Será usado o modelo:

- `intfloat/multilingual-e5-small`

### Justificativa da escolha
Esse modelo foi escolhido porque:
- é open-source;
- possui suporte multilíngue;
- é compatível com português;
- é mais leve e adequado para prototipação em ambiente gratuito;
- permite evolução futura para uma arquitetura totalmente local.

### Texto escolhido para vetorização
Nesta fase, o campo principal a ser vetorizado será:

- `texto_contextualizado`

### Justificativa
A escolha de `texto_contextualizado` foi feita porque ele preserva melhor o sentido jurídico do chunk dentro da norma, o que tende a melhorar a recuperação semântica.

### Métrica vetorial escolhida
Nesta fase será adotada a métrica:

- `cosine distance`

### Justificativa matemática
A distância cosseno é adequada para comparar embeddings semânticos porque mede a similaridade angular entre vetores, reduzindo o impacto da magnitude absoluta e favorecendo a comparação por direção semântica.

In [50]:
# Fase 5.4 - Instalação das bibliotecas para embeddings locais
#
# Esta etapa instala a biblioteca sentence-transformers, que será usada
# para gerar embeddings localmente, sem uso de API externa.
#
# Também instalamos o torch explicitamente para garantir compatibilidade
# com o carregamento do modelo local na sessão atual.

!pip install -q sentence-transformers torch

In [51]:
# Fase 5.4 - Carregamento do modelo local de embeddings
#
# Nesta célula:
# - carregamos o modelo local escolhido;
# - mantemos a decisão explícita em uma variável de configuração;
# - deixamos o notebook mais claro para auditoria e manutenção futura.

from sentence_transformers import SentenceTransformer

EMBEDDING_CONFIG = {
    "modelo_nome": "intfloat/multilingual-e5-small",
    "campo_textual_escolhido": "texto_contextualizado",
    "metrica_vetorial": "cosine_distance",
    "modo_execucao": "local_open_source"
}

modelo_embedding = SentenceTransformer(EMBEDDING_CONFIG["modelo_nome"])

print("Configuração de embeddings carregada com sucesso:")
print(json.dumps(EMBEDDING_CONFIG, ensure_ascii=False, indent=2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Configuração de embeddings carregada com sucesso:
{
  "modelo_nome": "intfloat/multilingual-e5-small",
  "campo_textual_escolhido": "texto_contextualizado",
  "metrica_vetorial": "cosine_distance",
  "modo_execucao": "local_open_source"
}


In [52]:
# Fase 5.4 - Teste do embedding local e identificação da dimensão vetorial
#
# O objetivo desta célula é:
# - confirmar que o modelo foi carregado corretamente;
# - gerar um vetor de teste localmente;
# - identificar a dimensão real do embedding.
#
# A dimensão será usada na próxima fase para criar a coluna vetorial
# no PostgreSQL com tamanho compatível.

texto_teste_embedding = chunks_norma[0][EMBEDDING_CONFIG["campo_textual_escolhido"]]

vetor_teste = modelo_embedding.encode(texto_teste_embedding, normalize_embeddings=True)

dimensao_embedding = len(vetor_teste)

print("Modelo de embeddings carregado:", EMBEDDING_CONFIG["modelo_nome"])
print("Campo textual vetorizado:", EMBEDDING_CONFIG["campo_textual_escolhido"])
print("Métrica vetorial escolhida:", EMBEDDING_CONFIG["metrica_vetorial"])
print("Dimensão do embedding:", dimensao_embedding)
print("Prévia do vetor (10 primeiros valores):")
print(vetor_teste[:10])

Modelo de embeddings carregado: intfloat/multilingual-e5-small
Campo textual vetorizado: texto_contextualizado
Métrica vetorial escolhida: cosine_distance
Dimensão do embedding: 384
Prévia do vetor (10 primeiros valores):
[ 0.0483726  -0.02083795 -0.0559895  -0.11054666  0.03369768 -0.0149202
  0.02963066  0.04543641  0.06939643  0.04635842]


## Teste de coerência da estratégia vetorial

Antes de criar a coluna vetorial no PostgreSQL, é importante validar:

- se o modelo local foi carregado com sucesso;
- se o embedding foi gerado sem erro;
- se a dimensão vetorial foi identificada corretamente.

Critério de aprovação: a dimensão do embedding deve ser maior que zero e o vetor precisa ser gerado sem uso de API externa.

In [53]:
# Fase 5.4 - Validação final da estratégia de embeddings locais

if dimensao_embedding <= 0:
    raise RuntimeError("A dimensão do embedding ficou inválida.")

print("Validação aprovada.")
print(f"Embeddings locais funcionando com dimensão {dimensao_embedding}.")
print("A estratégia vetorial está pronta para a criação da coluna vector no PostgreSQL.")

Validação aprovada.
Embeddings locais funcionando com dimensão 384.
A estratégia vetorial está pronta para a criação da coluna vector no PostgreSQL.


## Fase 5.5 — Adição da coluna vetorial e gravação dos embeddings no PostgreSQL

Nesta fase, a tabela relacional `chunks_juridicos` será preparada para armazenar vetores semânticos.

### Objetivo técnico
Adicionar ao PostgreSQL a camada vetorial do projeto, registrando:

- o vetor do chunk (`embedding`);
- o modelo utilizado para gerar o vetor;
- o campo textual usado no embedding;
- a informação de normalização;
- o instante em que o embedding foi gravado.

### Ferramentas usadas nesta fase
- **PostgreSQL**: banco de dados principal do projeto;
- **pgvector**: extensão vetorial do PostgreSQL;
- **sentence-transformers**: geração local dos embeddings;
- **psycopg2**: conexão Python com PostgreSQL;
- **pgvector para Python**: adaptação do tipo `vector` para escrita e leitura via psycopg2.

### Decisão técnica importante
Os embeddings gerados serão **normalizados** antes da gravação.

### Justificativa matemática
Como a métrica escolhida foi `cosine_distance`, a normalização dos vetores torna a comparação semântica mais consistente, reduzindo o efeito da magnitude absoluta e favorecendo a comparação angular entre embeddings.

### Resultado esperado
Ao final desta fase, todos os chunks jurídicos deverão possuir embedding gravado no PostgreSQL.

In [54]:
# Fase 5.5 - Instalação do suporte Python ao tipo vetorial do pgvector
#
# Embora a extensão pgvector já tenha sido instalada no PostgreSQL,
# nesta célula instalamos também o pacote Python correspondente.
#
# Finalidade:
# - permitir que vetores sejam enviados ao PostgreSQL a partir do Python;
# - permitir leitura futura de vetores via psycopg2 de forma compatível.

!pip install -q pgvector

In [55]:
# Fase 5.5 - Conexão PostgreSQL com suporte ao tipo vector
#
# Esta função encapsula a conexão com o PostgreSQL e registra o tipo vetorial
# do pgvector no psycopg2.
#
# Isso é necessário para que o Python consiga:
# - gravar embeddings na coluna vector;
# - ler embeddings futuramente sem tratamento manual de tipo.

from pgvector.psycopg2 import register_vector

def conectar_postgres_vetorial(config: dict):
    """
    Abre conexão com o PostgreSQL e registra suporte ao tipo vector
    da extensão pgvector.
    """
    conn = psycopg2.connect(**config)
    register_vector(conn)
    return conn

print("Função de conexão vetorial carregada com sucesso.")

Função de conexão vetorial carregada com sucesso.


## Expansão da tabela relacional para camada vetorial

Antes de gravar os embeddings, a tabela `chunks_juridicos` será ampliada com colunas específicas para a camada vetorial.

### Novas colunas desta fase
- `embedding` — vetor do chunk;
- `embedding_modelo` — nome do modelo que gerou o vetor;
- `embedding_campo_textual` — campo textual usado na vetorização;
- `embedding_normalizado` — informa se houve normalização;
- `embedding_atualizado_em` — timestamp da gravação vetorial.

### Benefício
Esse desenho permite:
- rastrear como cada vetor foi gerado;
- rerodar embeddings no futuro sem perder histórico metodológico;
- manter clareza técnica para auditoria e relatório.

In [56]:
# Fase 5.5 - Adição das colunas vetoriais na tabela chunks_juridicos
#
# Nesta etapa:
# - a coluna principal de vetor é criada com dimensão compatível (384);
# - colunas auxiliares de auditoria vetorial também são adicionadas.
#
# A dimensão usada vem da fase anterior, onde foi identificada a saída real
# do modelo de embeddings local.

from datetime import datetime, timezone

def adicionar_colunas_vetoriais(config: dict, dimensao: int) -> None:
    """
    Adiciona à tabela chunks_juridicos as colunas necessárias para armazenar
    embeddings e metadados da vetorização.
    """
    sql = f"""
    ALTER TABLE chunks_juridicos
        ADD COLUMN IF NOT EXISTS embedding vector({dimensao}),
        ADD COLUMN IF NOT EXISTS embedding_modelo TEXT,
        ADD COLUMN IF NOT EXISTS embedding_campo_textual TEXT,
        ADD COLUMN IF NOT EXISTS embedding_normalizado BOOLEAN,
        ADD COLUMN IF NOT EXISTS embedding_atualizado_em TIMESTAMPTZ;
    """

    with conectar_postgres_vetorial(config) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()

adicionar_colunas_vetoriais(PG_CONFIG, dimensao_embedding)

print(f"Colunas vetoriais adicionadas com sucesso com dimensão {dimensao_embedding}.")

Colunas vetoriais adicionadas com sucesso com dimensão 384.


In [57]:
# Fase 5.5 - Leitura dos chunks a partir do PostgreSQL para vetorização
#
# Nesta etapa, os textos são lidos diretamente do banco.
# Isso garante que os embeddings sejam gerados sobre a base persistida,
# e não apenas sobre objetos em memória da sessão Python.
#
# Estratégia textual:
# - o campo principal é texto_contextualizado;
# - se ele estiver vazio, usa-se texto_chunk como fallback.

def obter_textos_para_embedding(config: dict) -> list[dict]:
    """
    Lê do PostgreSQL os chunks que serão vetorizados, preservando a ordem.
    """
    sql = """
    SELECT
        chunk_id,
        ordem_chunk,
        COALESCE(NULLIF(texto_contextualizado, ''), texto_chunk) AS texto_para_embedding
    FROM chunks_juridicos
    ORDER BY ordem_chunk ASC;
    """

    with conectar_postgres_vetorial(config) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(sql)
            return cur.fetchall()

registros_para_embedding_db = obter_textos_para_embedding(PG_CONFIG)

print("Quantidade de registros lidos do PostgreSQL para embedding:", len(registros_para_embedding_db))
print("\nPrévia do primeiro registro lido:")
print(json.dumps(dict(registros_para_embedding_db[0]), ensure_ascii=False, indent=2)[:2000])

Quantidade de registros lidos do PostgreSQL para embedding: 254

Prévia do primeiro registro lido:
{
  "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__chunk_0",
  "ordem_chunk": 0,
  "texto_para_embedding": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990 Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990 Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte lei:"
}


## Geração dos embeddings em lote

Nesta etapa, os embeddings serão gerados localmente em lote.

### Motivo da estratégia em lote
A vetorização em lote melhora:
- desempenho;
- uso de memória;
- organização do pipeline.

### Observação metodológica
Nesta fase, o objetivo ainda não é fazer busca vetorial.
O objetivo é apenas **materializar a camada vetorial no banco**, preparando o sistema para a fase seguinte de consulta semântica.

In [58]:
# Fase 5.5 - Geração local dos embeddings em lote
#
# Esta célula usa o modelo local carregado na fase anterior para gerar
# os embeddings dos chunks lidos do banco.
#
# normalize_embeddings=True é mantido para coerência com a escolha de
# cosine_distance.

textos_para_embedding = [r["texto_para_embedding"] for r in registros_para_embedding_db]
chunk_ids_para_embedding = [r["chunk_id"] for r in registros_para_embedding_db]

embeddings_lote = modelo_embedding.encode(
    textos_para_embedding,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings gerados com sucesso.")
print("Quantidade de embeddings:", len(embeddings_lote))
print("Dimensão do primeiro embedding:", len(embeddings_lote[0]))

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embeddings gerados com sucesso.
Quantidade de embeddings: 254
Dimensão do primeiro embedding: 384


In [59]:
# Fase 5.5 - Gravação dos embeddings no PostgreSQL
#
# Esta célula associa cada embedding ao seu respectivo chunk_id
# e grava os vetores na tabela chunks_juridicos.
#
# Também registra:
# - o nome do modelo;
# - o campo textual vetorizado;
# - a informação de normalização;
# - o timestamp da gravação.

from psycopg2.extras import execute_batch

def gravar_embeddings_no_postgres(
    config: dict,
    chunk_ids: list[str],
    embeddings,
    embedding_config: dict
) -> int:
    """
    Atualiza a tabela chunks_juridicos com os embeddings gerados localmente.
    """
    sql = """
    UPDATE chunks_juridicos
    SET
        embedding = %s,
        embedding_modelo = %s,
        embedding_campo_textual = %s,
        embedding_normalizado = %s,
        embedding_atualizado_em = %s
    WHERE chunk_id = %s;
    """

    atualizado_em = datetime.now(timezone.utc)

    dados_update = [
        (
            embeddings[i],
            embedding_config["modelo_nome"],
            embedding_config["campo_textual_escolhido"],
            True,
            atualizado_em,
            chunk_ids[i]
        )
        for i in range(len(chunk_ids))
    ]

    with conectar_postgres_vetorial(config) as conn:
        with conn.cursor() as cur:
            execute_batch(cur, sql, dados_update, page_size=100)
        conn.commit()

    return len(dados_update)

quantidade_embeddings_gravados = gravar_embeddings_no_postgres(
    PG_CONFIG,
    chunk_ids_para_embedding,
    embeddings_lote,
    EMBEDDING_CONFIG
)

print("Gravação vetorial concluída com sucesso.")
print("Quantidade de embeddings gravados:", quantidade_embeddings_gravados)

Gravação vetorial concluída com sucesso.
Quantidade de embeddings gravados: 254


## Fase 5.5.A — Validação da camada vetorial gravada

Após a gravação, será feita uma validação da camada vetorial.

### Objetivo da validação
Confirmar:
- quantos registros possuem embedding;
- se a dimensão vetorial gravada está correta;
- se o nome do modelo foi registrado;
- se a camada vetorial está pronta para a fase de busca semântica.

In [60]:
# Fase 5.5.A - Validação da camada vetorial gravada
#
# Esta célula confere se os embeddings foram realmente persistidos na tabela
# chunks_juridicos e se a estrutura vetorial está pronta para consulta futura.

def validar_campos_vetoriais(config: dict) -> dict:
    with conectar_postgres_vetorial(config) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("""
                SELECT
                    COUNT(*) AS total,
                    COUNT(embedding) AS total_com_embedding
                FROM chunks_juridicos;
            """)
            resumo = cur.fetchone()

            cur.execute("""
                SELECT
                    chunk_id,
                    embedding,
                    embedding_modelo,
                    embedding_campo_textual,
                    embedding_normalizado
                FROM chunks_juridicos
                WHERE embedding IS NOT NULL
                ORDER BY ordem_chunk ASC
                LIMIT 1;
            """)
            amostra = cur.fetchone()

    return {
        "total_registros": resumo["total"],
        "total_com_embedding": resumo["total_com_embedding"],
        "amostra_chunk_id": amostra["chunk_id"] if amostra else None,
        "amostra_dimensao_embedding": len(amostra["embedding"]) if amostra and amostra["embedding"] is not None else None,
        "amostra_modelo": amostra["embedding_modelo"] if amostra else None,
        "amostra_campo_textual": amostra["embedding_campo_textual"] if amostra else None,
        "amostra_normalizado": amostra["embedding_normalizado"] if amostra else None
    }

resultado_validacao_vetorial = validar_campos_vetoriais(PG_CONFIG)

print("Resultado da validação vetorial:")
print(json.dumps(resultado_validacao_vetorial, ensure_ascii=False, indent=2))

Resultado da validação vetorial:
{
  "total_registros": 254,
  "total_com_embedding": 254,
  "amostra_chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__chunk_0",
  "amostra_dimensao_embedding": 384,
  "amostra_modelo": "intfloat/multilingual-e5-small",
  "amostra_campo_textual": "texto_contextualizado",
  "amostra_normalizado": true
}


In [61]:
# Fase 5.5.A - Critério final de aprovação da gravação vetorial

if resultado_validacao_vetorial["total_registros"] != len(chunks_norma):
    raise RuntimeError("O total de registros no banco não bate com a quantidade de chunks.")

if resultado_validacao_vetorial["total_com_embedding"] != len(chunks_norma):
    raise RuntimeError("Nem todos os chunks receberam embedding.")

if resultado_validacao_vetorial["amostra_dimensao_embedding"] != dimensao_embedding:
    raise RuntimeError("A dimensão do embedding gravado não bate com a dimensão esperada.")

print("Validação aprovada.")
print("A camada vetorial foi gravada com sucesso no PostgreSQL.")
print("O banco está pronto para a fase de consulta vetorial semântica.")

Validação aprovada.
A camada vetorial foi gravada com sucesso no PostgreSQL.
O banco está pronto para a fase de consulta vetorial semântica.


## Critério de aprovação da Fase 5.5

A fase será considerada aprovada se:

- a coluna vetorial for criada com a dimensão correta;
- todos os chunks receberem embedding;
- o modelo e o campo textual forem registrados;
- a validação final confirmar consistência da camada vetorial.

Se isso ocorrer, a base vetorial estará pronta para a próxima fase de busca semântica no PostgreSQL.

## Fase 5.5.B — Persistência do inteiro teor integral no PostgreSQL

Nesta fase, além dos chunks jurídicos já persistidos, o **inteiro teor integral da norma** também será salvo no PostgreSQL.

### Objetivo técnico
Garantir que o banco contenha dois níveis complementares de representação da norma:

1. **texto integral completo** — preserva o documento jurídico como fonte primária;
2. **chunks jurídicos vetorizados** — suportam busca semântica e recuperação contextual.

### Motivo desta decisão
O projeto não deve depender apenas de fragmentos resumidos ou vetorizados.

Ao manter o inteiro teor integral no banco, o sistema passa a ter:
- rastreabilidade documental;
- preservação do conteúdo completo aprovado pelo cliente;
- possibilidade futura de uso por IAs locais/open-source com base na norma inteira.

### Estratégia adotada
Será criada uma tabela própria para o texto integral da norma, separada da tabela de chunks.

### Benefício arquitetural
Essa separação melhora:
- organização do banco;
- clareza metodológica;
- facilidade de auditoria;
- reutilização futura da base por diferentes estratégias de IA.

In [62]:
# Fase 5.5.B - Extração estruturada do inteiro teor integral a partir do JSON canônico
#
# Esta versão é mais robusta que a anterior.
#
# Estratégia adotada:
# 1. tenta localizar o texto integral diretamente no JSON canônico;
# 2. se não encontrar, reconstrói o inteiro teor juntando os chunks em ordem;
# 3. preserva também o JSON canônico completo para auditoria.
#
# Isso garante que o inteiro teor integral continue existindo como documento
# principal no banco, mesmo que o JSON canônico não tenha um campo único
# chamado "texto_integral" ou "texto_preferido".

from psycopg2.extras import Json

def obter_primeiro_preenchido(*valores):
    """
    Retorna o primeiro valor não vazio dentre os candidatos recebidos.
    Considera vazio: None, string vazia, lista vazia e dict vazio.
    """
    for valor in valores:
        if valor not in (None, "", [], {}):
            return valor
    return None

def reconstruir_texto_integral_a_partir_dos_chunks(chunks_norma: list[dict]) -> str:
    """
    Reconstrói o inteiro teor integral da norma a partir dos chunks já gerados,
    preservando a ordem jurídica original.

    Estratégia:
    - ordena os chunks por ordem_chunk;
    - usa texto_chunk como base textual principal;
    - concatena os trechos em sequência para remontar o documento completo.
    """
    chunks_ordenados = sorted(chunks_norma, key=lambda c: c.get("ordem_chunk", 0))

    partes = []
    for chunk in chunks_ordenados:
        texto = (chunk.get("texto_chunk") or "").strip()
        if texto:
            partes.append(texto)

    texto_reconstruido = "\n\n".join(partes).strip()
    return texto_reconstruido

def montar_registro_norma_integral(json_canonico: dict, chunks_norma: list[dict]) -> dict:
    """
    Monta um registro único da norma integral para persistência no PostgreSQL.

    Fontes utilizadas:
    - JSON canônico, quando o texto integral existir de forma explícita;
    - reconstrução a partir dos chunks, como fallback robusto.
    """
    canonical = json_canonico.get("canonical", {})
    texto_info = json_canonico.get("texto", {})
    primeiro_chunk = chunks_norma[0] if chunks_norma else {}

    urn = obter_primeiro_preenchido(
        canonical.get("urn"),
        json_canonico.get("urn"),
        primeiro_chunk.get("urn")
    )

    titulo_norma = obter_primeiro_preenchido(
        canonical.get("titulo_norma"),
        json_canonico.get("titulo_norma"),
        primeiro_chunk.get("titulo_norma")
    )

    ementa = obter_primeiro_preenchido(
        canonical.get("ementa"),
        json_canonico.get("ementa"),
        primeiro_chunk.get("ementa")
    )

    fonte_preferida = obter_primeiro_preenchido(
        texto_info.get("fonte_preferida"),
        json_canonico.get("fonte_preferida"),
        primeiro_chunk.get("fonte")
    )

    url_origem = obter_primeiro_preenchido(
        texto_info.get("url_origem"),
        json_canonico.get("url_origem"),
        primeiro_chunk.get("url_origem")
    )

    coletado_em = obter_primeiro_preenchido(
        json_canonico.get("coletado_em"),
        texto_info.get("coletado_em"),
        primeiro_chunk.get("coletado_em")
    )

    pipeline_version = obter_primeiro_preenchido(
        json_canonico.get("pipeline_version"),
        texto_info.get("pipeline_version"),
        primeiro_chunk.get("pipeline_version")
    )

    # Tentativa 1: localizar o texto integral diretamente no JSON canônico
    texto_integral = obter_primeiro_preenchido(
        texto_info.get("texto_preferido"),
        texto_info.get("texto_integral"),
        json_canonico.get("texto_preferido"),
        json_canonico.get("texto_integral")
    )

    # Tentativa 2: reconstruir o texto integral a partir dos chunks, se necessário
    if not texto_integral:
        texto_integral = reconstruir_texto_integral_a_partir_dos_chunks(chunks_norma)

    if not urn:
        raise RuntimeError("Não foi possível identificar a URN da norma integral.")

    if not texto_integral:
        raise RuntimeError(
            "Não foi possível localizar nem reconstruir o texto integral da norma."
        )

    return {
        "urn": urn,
        "titulo_norma": titulo_norma,
        "ementa": ementa,
        "fonte_preferida": fonte_preferida,
        "url_origem": url_origem,
        "texto_integral": texto_integral,
        "quantidade_caracteres": len(texto_integral),
        "coletado_em": coletado_em,
        "pipeline_version": pipeline_version,
        "json_canonico": Json(json_canonico)
    }

registro_norma_integral = montar_registro_norma_integral(json_canonico, chunks_norma)

print("Registro da norma integral preparado com sucesso.")
print("URN:", registro_norma_integral["urn"])
print("Fonte preferida:", registro_norma_integral["fonte_preferida"])
print("Quantidade de caracteres do texto integral:", registro_norma_integral["quantidade_caracteres"])
print("\nPrévia do início do texto integral:")
print(registro_norma_integral["texto_integral"][:700])

Registro da norma integral preparado com sucesso.
URN: urn:lex:br:federal:lei:1990-12-11;8112
Fonte preferida: camara
Quantidade de caracteres do texto integral: 88408

Prévia do início do texto integral:
LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990 Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte lei:

Art. 1º Esta Lei institui o regime jurídico dos servidores públicos civis da União, das autarquias, inclusive as em regime especial, e das fundações públicas federais.

Art. 2º Para os efeitos desta Lei, servidor é a pessoa legalmente investida em cargo público.

Art. 3º Cargo público é o conjunto de atribuições e responsabilidades previstas na estrutura organizacional que devem ser cometidas a um servidor. Parágrafo único. Os cargos pú


## Tabela da norma integral

Nesta etapa será criada uma tabela específica para armazenar a norma completa.

### Tabela criada
`normas_integras`

### Finalidade
Armazenar:
- a URN da norma;
- título;
- ementa;
- fonte preferida;
- URL de origem;
- texto integral completo;
- tamanho do texto;
- timestamp de coleta;
- versão do pipeline;
- JSON canônico completo.

### Observação
Essa tabela não substitui os chunks jurídicos.
Ela complementa a arquitetura do banco e preserva o documento completo como fonte primária.

In [63]:
# Fase 5.5.B - Criação da tabela da norma integral e persistência do inteiro teor
#
# Nesta célula:
# - criamos a tabela normas_integras, caso ainda não exista;
# - usamos URN como chave única da norma;
# - aplicamos UPSERT para permitir reexecução segura no notebook.
#
# Benefício:
# - o inteiro teor passa a existir de forma persistida no PostgreSQL;
# - isso prepara o sistema para estratégias futuras com IA local.

def criar_tabela_normas_integras(config: dict) -> None:
    sql = """
    CREATE TABLE IF NOT EXISTS normas_integras (
        id SERIAL PRIMARY KEY,
        urn TEXT UNIQUE NOT NULL,
        titulo_norma TEXT,
        ementa TEXT,
        fonte_preferida TEXT,
        url_origem TEXT,
        texto_integral TEXT NOT NULL,
        quantidade_caracteres INTEGER NOT NULL,
        coletado_em TEXT,
        pipeline_version TEXT,
        json_canonico JSONB NOT NULL
    );
    """

    with psycopg2.connect(**config) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()

def salvar_norma_integral_no_postgres(config: dict, registro: dict) -> None:
    sql = """
    INSERT INTO normas_integras (
        urn,
        titulo_norma,
        ementa,
        fonte_preferida,
        url_origem,
        texto_integral,
        quantidade_caracteres,
        coletado_em,
        pipeline_version,
        json_canonico
    )
    VALUES (
        %(urn)s,
        %(titulo_norma)s,
        %(ementa)s,
        %(fonte_preferida)s,
        %(url_origem)s,
        %(texto_integral)s,
        %(quantidade_caracteres)s,
        %(coletado_em)s,
        %(pipeline_version)s,
        %(json_canonico)s
    )
    ON CONFLICT (urn)
    DO UPDATE SET
        titulo_norma = EXCLUDED.titulo_norma,
        ementa = EXCLUDED.ementa,
        fonte_preferida = EXCLUDED.fonte_preferida,
        url_origem = EXCLUDED.url_origem,
        texto_integral = EXCLUDED.texto_integral,
        quantidade_caracteres = EXCLUDED.quantidade_caracteres,
        coletado_em = EXCLUDED.coletado_em,
        pipeline_version = EXCLUDED.pipeline_version,
        json_canonico = EXCLUDED.json_canonico;
    """

    with psycopg2.connect(**config) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, registro)
        conn.commit()

criar_tabela_normas_integras(PG_CONFIG)
salvar_norma_integral_no_postgres(PG_CONFIG, registro_norma_integral)

print("Tabela 'normas_integras' criada/verificada com sucesso.")
print("Inteiro teor integral salvo no PostgreSQL com sucesso.")

Tabela 'normas_integras' criada/verificada com sucesso.
Inteiro teor integral salvo no PostgreSQL com sucesso.


## Fase 5.5.C — Validação da persistência do inteiro teor integral

Após a gravação, será feita uma validação da tabela `normas_integras`.

### Objetivo da validação
Confirmar:
- se a norma foi realmente persistida;
- se a URN ficou correta;
- se a fonte preferida foi registrada;
- se o texto integral possui tamanho compatível;
- se a base agora preserva o documento completo.

In [64]:
# Fase 5.5.C - Validação da persistência do inteiro teor integral
#
# Esta célula consulta a tabela normas_integras para verificar
# se o texto completo foi salvo corretamente e se os principais metadados
# foram preservados.

def validar_norma_integral(config: dict, urn: str) -> dict:
    sql = """
    SELECT
        urn,
        titulo_norma,
        fonte_preferida,
        quantidade_caracteres,
        LEFT(texto_integral, 500) AS amostra_inicio_texto
    FROM normas_integras
    WHERE urn = %s;
    """

    with psycopg2.connect(**config) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(sql, (urn,))
            resultado = cur.fetchone()

    return dict(resultado) if resultado else {}

resultado_validacao_norma_integral = validar_norma_integral(
    PG_CONFIG,
    registro_norma_integral["urn"]
)

print("Resultado da validação da norma integral:")
print(json.dumps(resultado_validacao_norma_integral, ensure_ascii=False, indent=2))

Resultado da validação da norma integral:
{
  "urn": "urn:lex:br:federal:lei:1990-12-11;8112",
  "titulo_norma": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990",
  "fonte_preferida": "camara",
  "quantidade_caracteres": 88408,
  "amostra_inicio_texto": "LEI Nº 8112, DE 11 DE DEZEMBRO DE 1990 Dispõe sobre o Regime Jurídico dos Servidores Públicos Civis da União, das autarquias e das fundações públicas federais O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte lei:\n\nArt. 1º Esta Lei institui o regime jurídico dos servidores públicos civis da União, das autarquias, inclusive as em regime especial, e das fundações públicas federais.\n\nArt. 2º Para os efeitos desta Lei, servidor é a pessoa legalmente inves"
}


In [65]:
# Fase 5.5.C - Critério final de aprovação da persistência do inteiro teor

if not resultado_validacao_norma_integral:
    raise RuntimeError("A norma integral não foi encontrada na tabela normas_integras.")

if resultado_validacao_norma_integral["quantidade_caracteres"] <= 0:
    raise RuntimeError("A norma integral foi salva, mas com tamanho inválido.")

print("Validação aprovada.")
print("O inteiro teor integral está persistido no PostgreSQL.")
print("O banco agora preserva tanto o documento completo quanto os chunks vetorizados.")

Validação aprovada.
O inteiro teor integral está persistido no PostgreSQL.
O banco agora preserva tanto o documento completo quanto os chunks vetorizados.


## Critério de aprovação da Fase 5.5.B / 5.5.C

A fase será considerada aprovada se:

- a tabela `normas_integras` for criada sem erro;
- a norma integral for persistida com a URN correta;
- o texto integral tiver tamanho maior que zero;
- a validação final confirmar que o banco preserva tanto o documento completo quanto os chunks vetorizados.

Se isso ocorrer, a arquitetura do banco estará alinhada com o requisito do cliente de manter o inteiro teor integral como base primária do sistema.

## Fase 5.6 — Consulta vetorial semântica no PostgreSQL

Nesta fase será validada a busca semântica sobre os chunks jurídicos já vetorizados no PostgreSQL.

### Objetivo técnico
Demonstrar que o banco vetorial já consegue:

- receber uma pergunta em linguagem natural;
- gerar o embedding local da pergunta;
- comparar a pergunta com os chunks vetorizados;
- retornar os trechos juridicamente mais próximos.

### Ferramentas envolvidas nesta fase
- **PostgreSQL**: armazenamento dos chunks e embeddings;
- **pgvector**: comparação vetorial dentro do banco;
- **sentence-transformers**: geração local do embedding da consulta;
- **psycopg2**: execução da consulta SQL pelo Python.

### Métrica usada nesta fase
A consulta vetorial será feita com **cosine distance**.

### Justificativa
Como os embeddings foram normalizados e a estratégia definida anteriormente usa `cosine_distance`, esta fase mantém coerência matemática com a modelagem adotada.

In [66]:
# Fase 5.6 - Geração do embedding da consulta em linguagem natural
#
# Esta função transforma a pergunta do usuário em um vetor compatível com os
# embeddings já gravados no PostgreSQL.
#
# A mesma estratégia usada nos chunks é mantida aqui:
# - mesmo modelo local;
# - mesma normalização;
# - mesma dimensão vetorial.
#
# Isso garante coerência entre:
# - vetor da consulta
# - vetores dos chunks armazenados

def gerar_embedding_consulta(pergunta: str, modelo_embedding) -> list[float]:
    """
    Gera o embedding local da pergunta em linguagem natural.
    """
    pergunta = (pergunta or "").strip()
    if not pergunta:
        raise ValueError("A pergunta da consulta vetorial não pode ser vazia.")

    vetor = modelo_embedding.encode(
        pergunta,
        normalize_embeddings=True
    )

    return vetor.tolist()

print("Função de embedding da consulta carregada com sucesso.")

Função de embedding da consulta carregada com sucesso.


## Consulta vetorial por similaridade semântica

Nesta etapa será executada uma busca vetorial real no PostgreSQL.

### Estratégia SQL
Será usada a operação vetorial do `pgvector` com `<=>`, que representa a distância cosseno entre vetores quando a estratégia adotada é cosine distance.

### Resultado esperado
A consulta deverá retornar os chunks mais próximos semanticamente da pergunta, preservando:

- `chunk_id`
- `tipo_bloco`
- `artigo`
- `ordem_chunk`
- `distancia`
- `texto_chunk`

In [67]:
# Fase 5.6 - Consulta vetorial semântica no PostgreSQL
#
# Esta função executa a busca vetorial propriamente dita.
#
# Estratégia:
# - recebe a pergunta;
# - gera o embedding localmente;
# - envia o vetor ao PostgreSQL;
# - compara contra a coluna embedding via pgvector;
# - retorna os top-k chunks mais próximos semanticamente.
#
# Observação:
# Quanto menor a distância retornada, maior a proximidade semântica.

def buscar_chunks_semanticos_postgres(
    config: dict,
    pergunta: str,
    modelo_embedding,
    top_k: int = 5
) -> list[dict]:
    """
    Realiza busca vetorial semântica no PostgreSQL usando cosine distance.
    """
    vetor_consulta = gerar_embedding_consulta(pergunta, modelo_embedding)

    sql = """
    SELECT
        chunk_id,
        tipo_bloco,
        artigo,
        ordem_chunk,
        texto_chunk,
        embedding <=> %s::vector AS distancia
    FROM chunks_juridicos
    WHERE embedding IS NOT NULL
    ORDER BY embedding <=> %s::vector ASC
    LIMIT %s;
    """

    with conectar_postgres_vetorial(config) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(sql, (vetor_consulta, vetor_consulta, top_k))
            resultados = cur.fetchall()

    return [dict(r) for r in resultados]

print("Função de busca vetorial carregada com sucesso.")

Função de busca vetorial carregada com sucesso.


## Teste 1 — Consulta vetorial sobre investidura em cargo público

Neste teste será usada uma pergunta simples de validação semântica.

### Pergunta de teste
"Quais são os requisitos para investidura em cargo público?"

### Critério de leitura do resultado
Os primeiros chunks retornados devem se aproximar dos trechos jurídicos ligados ao tema da investidura, especialmente os artigos iniciais da Lei 8.112.

In [68]:
# Fase 5.6 - Teste vetorial com pergunta jurídica
#
# Esta célula executa uma consulta vetorial real e imprime os resultados
# de forma legível para validação manual da recuperação semântica.

pergunta_teste_vetorial_1 = "Quais são os requisitos para investidura em cargo público?"

resultados_consulta_vetorial_1 = buscar_chunks_semanticos_postgres(
    config=PG_CONFIG,
    pergunta=pergunta_teste_vetorial_1,
    modelo_embedding=modelo_embedding,
    top_k=5
)

print("Pergunta de teste:")
print(pergunta_teste_vetorial_1)
print("\nResultados vetoriais encontrados:", len(resultados_consulta_vetorial_1))

for i, item in enumerate(resultados_consulta_vetorial_1, start=1):
    print(f"\nResultado #{i}")
    print("chunk_id:", item["chunk_id"])
    print("tipo_bloco:", item["tipo_bloco"])
    print("artigo:", item["artigo"])
    print("ordem_chunk:", item["ordem_chunk"])
    print("distancia:", item["distancia"])
    print("texto_chunk:", item["texto_chunk"][:500])

Pergunta de teste:
Quais são os requisitos para investidura em cargo público?

Resultados vetoriais encontrados: 5

Resultado #1
chunk_id: urn_lex_br_federal_lei_1990_12_11_8112__art_5__chunk_5
tipo_bloco: artigo
artigo: 5
ordem_chunk: 5
distancia: 0.11678600311279297
texto_chunk: Art. 5º São requisitos básicos para investidura em cargo público: I - a nacionalidade brasileira; II - o gozo dos direitos políticos; III - a quitação com as obrigações militares e eleitorais; IV - o nível de escolaridade exigido para o exercício do cargo; V - a idade mínima de dezoito anos; VI - aptidão física e mental. § 1º As atribuições do cargo podem justificar a exigência de outros requisitos estabelecidos em lei. § 2º Às pessoas portadoras de deficiência é assegurado o direito de se inscr

Resultado #2
chunk_id: urn_lex_br_federal_lei_1990_12_11_8112__art_137__chunk_137
tipo_bloco: artigo
artigo: 137
ordem_chunk: 137
distancia: 0.14193010330200195
texto_chunk: Art. 137. A demissão, ou a destituição de 

In [70]:
# Fase 5.6 - Resumo estruturado da consulta vetorial
#
# Esta célula transforma o resultado bruto da busca em uma estrutura mais
# resumida, útil para auditoria e interpretação.

resumo_consulta_vetorial_1 = [
    {
        "ranking": i + 1,
        "chunk_id": item["chunk_id"],
        "tipo_bloco": item["tipo_bloco"],
        "artigo": item["artigo"],
        "ordem_chunk": item["ordem_chunk"],
        "distancia": round(float(item["distancia"]), 6)
    }
    for i, item in enumerate(resultados_consulta_vetorial_1)
]

print("Resumo da consulta vetorial:")
print(json.dumps(resumo_consulta_vetorial_1, ensure_ascii=False, indent=2))

Resumo da consulta vetorial:
[
  {
    "ranking": 1,
    "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__art_5__chunk_5",
    "tipo_bloco": "artigo",
    "artigo": "5",
    "ordem_chunk": 5,
    "distancia": 0.116786
  },
  {
    "ranking": 2,
    "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__art_137__chunk_137",
    "tipo_bloco": "artigo",
    "artigo": "137",
    "ordem_chunk": 137,
    "distancia": 0.14193
  },
  {
    "ranking": 3,
    "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__art_7__chunk_7",
    "tipo_bloco": "artigo",
    "artigo": "7",
    "ordem_chunk": 7,
    "distancia": 0.142165
  },
  {
    "ranking": 4,
    "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__art_2__chunk_2",
    "tipo_bloco": "artigo",
    "artigo": "2",
    "ordem_chunk": 2,
    "distancia": 0.14594
  },
  {
    "ranking": 5,
    "chunk_id": "urn_lex_br_federal_lei_1990_12_11_8112__art_14__chunk_14",
    "tipo_bloco": "artigo",
    "artigo": "14",
    "ordem_chunk": 14,
    "dista

## Critério de aprovação da Fase 5.6

A fase será considerada aprovada se:

- a consulta vetorial for executada sem erro;
- forem retornados resultados com `distancia` calculada;
- os chunks retornados forem semanticamente coerentes com a pergunta de teste.

Se isso ocorrer, a fase de banco vetorial poderá ser considerada funcionalmente validada.

In [71]:
# Fase 5.6 - Validação final da consulta vetorial semântica

if not resultados_consulta_vetorial_1:
    raise RuntimeError("A consulta vetorial não retornou resultados.")

if any(item["distancia"] is None for item in resultados_consulta_vetorial_1):
    raise RuntimeError("Pelo menos um resultado veio sem distância calculada.")

print("Validação aprovada.")
print("A consulta vetorial semântica no PostgreSQL está funcionando.")
print("A fase de banco vetorial foi validada com sucesso.")

Validação aprovada.
A consulta vetorial semântica no PostgreSQL está funcionando.
A fase de banco vetorial foi validada com sucesso.


## Fase 5.6.A — Otimização e indexação da camada vetorial

Nesta subfase será feito o acabamento técnico da camada de banco vetorial, com criação de índices apropriados para consultas futuras.

### Objetivo técnico
Preparar o banco para:
- escalar melhor em volume de chunks;
- manter consultas semânticas mais eficientes;
- facilitar consultas auxiliares por artigo e ordem dos chunks.

### Índices criados nesta etapa
- índice vetorial HNSW na coluna `embedding`;
- índice relacional em `artigo`;
- índice relacional em `ordem_chunk`.

### Justificativa técnica
Mesmo que a base atual ainda seja pequena, a criação dos índices fecha a fase de banco de dados de forma mais profissional e deixa a estrutura preparada para evolução futura.

In [72]:
# Fase 5.6.A - Criação de índices para a camada vetorial e relacional
#
# Esta célula cria os índices mais importantes da fase de banco:
# 1. índice vetorial HNSW para busca por similaridade em cosine distance;
# 2. índice relacional no campo artigo;
# 3. índice relacional no campo ordem_chunk.
#
# Observação importante:
# Em uma base pequena como a atual, o ganho de desempenho pode não ser perceptível.
# Ainda assim, os índices deixam a arquitetura pronta para escalabilidade futura.

def criar_indices_chunks_juridicos(config: dict) -> None:
    sqls = [
        """
        CREATE INDEX IF NOT EXISTS idx_chunks_embedding_hnsw
        ON chunks_juridicos
        USING hnsw (embedding vector_cosine_ops);
        """,
        """
        CREATE INDEX IF NOT EXISTS idx_chunks_artigo
        ON chunks_juridicos (artigo);
        """,
        """
        CREATE INDEX IF NOT EXISTS idx_chunks_ordem_chunk
        ON chunks_juridicos (ordem_chunk);
        """
    ]

    with conectar_postgres_vetorial(config) as conn:
        with conn.cursor() as cur:
            for sql in sqls:
                cur.execute(sql)
        conn.commit()

criar_indices_chunks_juridicos(PG_CONFIG)

print("Índices da camada vetorial e relacional criados com sucesso.")

Índices da camada vetorial e relacional criados com sucesso.


In [74]:
# Fase 5.6.A - Validação dos índices criados
#
# Esta célula consulta o catálogo do PostgreSQL para listar os índices
# existentes na tabela chunks_juridicos.

def listar_indices_tabela(config: dict, nome_tabela: str) -> list[dict]:
    sql = """
    SELECT
        indexname,
        indexdef
    FROM pg_indexes
    WHERE tablename = %s
    ORDER BY indexname;
    """

    with psycopg2.connect(**config) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(sql, (nome_tabela,))
            return cur.fetchall()

indices_chunks_juridicos = listar_indices_tabela(PG_CONFIG, "chunks_juridicos")

print("Índices encontrados na tabela 'chunks_juridicos':\n")
for item in indices_chunks_juridicos:
    print("-", item["indexname"])
    print(" ", item["indexdef"])
    print()

Índices encontrados na tabela 'chunks_juridicos':

- chunks_juridicos_chunk_id_key
  CREATE UNIQUE INDEX chunks_juridicos_chunk_id_key ON public.chunks_juridicos USING btree (chunk_id)

- chunks_juridicos_pkey
  CREATE UNIQUE INDEX chunks_juridicos_pkey ON public.chunks_juridicos USING btree (id)

- idx_chunks_artigo
  CREATE INDEX idx_chunks_artigo ON public.chunks_juridicos USING btree (artigo)

- idx_chunks_embedding_hnsw
  CREATE INDEX idx_chunks_embedding_hnsw ON public.chunks_juridicos USING hnsw (embedding vector_cosine_ops)

- idx_chunks_ordem_chunk
  CREATE INDEX idx_chunks_ordem_chunk ON public.chunks_juridicos USING btree (ordem_chunk)



## Critério de aprovação da Fase 5.6.A

A subfase será considerada aprovada se:

- o índice vetorial HNSW for criado com sucesso;
- os índices relacionais forem listados corretamente;
- a tabela `chunks_juridicos` ficar preparada para consultas vetoriais e relacionais futuras.

### Observação metodológica
Em bases pequenas, o PostgreSQL pode ainda optar por planos simples de execução.
Isso não significa falha da indexação, apenas que o custo da consulta ainda é baixo.